# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



## Phase 0: Examine the Dataset (Meta)

SQL in sqlite3

In [2]:
import sqlite3 as db
import pandas as pd
import contextlib

In [3]:
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [4]:
! sqlite3 --help

Usage: sqlite3 [OPTIONS] [FILENAME [SQL]]
FILENAME is the name of an SQLite database. A new database is created
if the file does not previously exist. Defaults to :memory:.
OPTIONS include:
   --                   treat no subsequent arguments as options
   -A ARGS...           run ".archive ARGS" and exit
   -append              append the database to the end of the file
   -ascii               set output mode to 'ascii'
   -bail                stop after hitting an error
   -batch               force batch I/O
   -box                 set output mode to 'box'
   -column              set output mode to 'column'
   -cmd COMMAND         run "COMMAND" before reading stdin
   -csv                 set output mode to 'csv'
   -deserialize         open the database using sqlite3_deserialize()
   -echo                print inputs before execution
   -init FILENAME       read/process named file
   -[no]header          turn headers on or off
   -help                show this message
   -html    

In [5]:
! sqlite3 --version

3.45.1 2024-01-30 16:01:20 e876e51a0ed5c5b3126f52e532044363a014bc594cfefa87ffb5b82257ccalt1 (64-bit)


In [6]:
# Download the dataset
%%bash
[ -f chinook.zip ] ||
  curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -l chinook.zip

Archive:  chinook.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   884736  2015-11-29 10:53   chinook.db
---------                     -------
   884736                     1 file


In [7]:
!ls -l

total 304
-rw-r--r-- 1 root root 305596 Nov 17 07:14 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [8]:
!rm -f chinook.db

In [9]:
!ls -l

total 304
-rw-r--r-- 1 root root 305596 Nov 17 07:14 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [10]:
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [11]:
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 17 07:14 chinook.zip
drwxr-xr-x 1 root root   4096 Oct 26 22:18 sample_data


In [12]:
# Get a list of the tables in the database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [13]:
# Show the schema for the entire database
%%script sqlite3 --column --header chinook.db
.schema

CREATE TABLE IF NOT EXISTS "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
);
CREATE TABLE sqlite_sequence(name,seq);
CREATE TABLE IF NOT EXISTS "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);
CREATE TABLE IF NOT EXISTS "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON 

## Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



### 1. Artists

In [239]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from artists
limit 5

ArtistId  Name             
--------  -----------------
1         AC/DC            
2         Accept           
3         Aerosmith        
4         Alanis Morissette
5         Alice In Chains  


In [240]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    ArtistId,
    Name
FROM artists
LIMIT 10;

ArtistId  Name                
--------  --------------------
1         AC/DC               
2         Accept              
3         Aerosmith           
4         Alanis Morissette   
5         Alice In Chains     
6         Antônio Carlos Jobim
7         Apocalyptica        
8         Audioslave          
9         BackBeat            
10        Billy Cobham        


In [241]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('artists') -- must have 'artists', not artists
PRAGMA table_info(artists);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    ArtistId  INTEGER        1                    1 
1    Name      NVARCHAR(120)  0                    0 


In [242]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('artists')
WHERE pk > 0;

name      type     pk
--------  -------  --
ArtistId  INTEGER  1 


In [243]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('artists');

In [244]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('artists')

total_columns
-------------
2            


In [245]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_artists
from artists

total_artists
-------------
275          


In [246]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM artists
WHERE Name IS NULL;

null_names
----------
0         


In [247]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM artists;

total_rows  null_artist_ids  null_names
----------  ---------------  ----------
275         0                0         


In [248]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM artists;

total_rows  rows_with_nulls
----------  ---------------
275         0              


In [249]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ArtistId IS NULL THEN 1 ELSE 0 END) AS null_artist_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN ArtistId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM artists;

total_rows  null_artist_ids  null_names  rows_with_nulls
----------  ---------------  ----------  ---------------
275         0                0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [250]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE Name LIKE 'A%';

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
1         AC/DC                                                                                
2         Accept                                                                               
3         Aerosmith                                                                            
4         Alanis Morissette                                                                    
5         Alice In Chains                                                                      
6         Antônio Carlos Jobim                                                                 
7         Apocalyptica                                                                         
8         Audioslave                                                                           
26        Azymuth                       

In [251]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE Name LIKE 'A_%';

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
1         AC/DC                                                                                
2         Accept                                                                               
3         Aerosmith                                                                            
4         Alanis Morissette                                                                    
5         Alice In Chains                                                                      
6         Antônio Carlos Jobim                                                                 
7         Apocalyptica                                                                         
8         Audioslave                                                                           
26        Azymuth                       

In [252]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE ArtistId BETWEEN 1 AND 50
  AND Name LIKE '%H%';

ArtistId  Name                           
--------  -------------------------------
3         Aerosmith                      
5         Alice In Chains                
10        Billy Cobham                   
12        Black Sabbath                  
17        Chico Buarque                  
18        Chico Science & Nação Zumbi    
23        Frank Zappa & Captain Beefheart
26        Azymuth                        
44        Kid Abelha                     
47        Hermeto Pascoal                
48        Barão Vermelho                 


In [253]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT ArtistId, Name
FROM artists
WHERE ArtistId = 1
   OR ArtistId = 2
   OR ArtistId = 3;

ArtistId  Name     
--------  ---------
1         AC/DC    
2         Accept   
3         Aerosmith


In [254]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM artists
LIMIT 5;

ArtistId  Name             
--------  -----------------
1         AC/DC            
2         Accept           
3         Aerosmith        
4         Alanis Morissette
5         Alice In Chains  


In [255]:
# Count unique artist names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_artist_names
FROM artists;

unique_artist_names
-------------------
275                


In [256]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    ArtistId,
    Name
FROM artists
ORDER BY Name ASC;

ArtistId  Name                                                                                 
--------  -------------------------------------------------------------------------------------
43        A Cor Do Som                                                                         
1         AC/DC                                                                                
230       Aaron Copland & London Symphony Orchestra                                            
202       Aaron Goldberg                                                                       
214       Academy of St. Martin in the Fields & Sir Neville Marriner                           
215       Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner          
222       Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair
257       Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart            
239       Academy of St. Martin in the F

In [257]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY occurrences DESC;

Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
Zeca Pagodinho                                                                         1          
Youssou N'Dour                                                                         1          
Yo-Yo Ma                                                                               1          
Yehudi Menuhin                                                                         1          
Xis                                                                                    1          
Wilhelm Kempff                                                                         1          
Whitesnake                                                                             1          
Vinícius E Qurteto Em Cy                                                               1          
Vinícius E

In [258]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM artists
GROUP BY Name
ORDER BY Name
LIMIT 10;

Name                                                                                   occurrences
-------------------------------------------------------------------------------------  -----------
A Cor Do Som                                                                           1          
AC/DC                                                                                  1          
Aaron Copland & London Symphony Orchestra                                              1          
Aaron Goldberg                                                                         1          
Academy of St. Martin in the Fields & Sir Neville Marriner                             1          
Academy of St. Martin in the Fields Chamber Ensemble & Sir Neville Marriner            1          
Academy of St. Martin in the Fields, John Birch, Sir Neville Marriner & Sylvia McNair  1          
Academy of St. Martin in the Fields, Sir Neville Marriner & Thurston Dart              1          
Academy of

In [259]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_artists,
    SUM(ArtistId) AS sum_artist_ids,
    AVG(ArtistId) AS average_artist_id,
    MAX(ArtistId) AS highest_artist_id,
    MIN(ArtistId) AS lowest_artist_id
FROM artists;

total_artists  sum_artist_ids  average_artist_id  highest_artist_id  lowest_artist_id
-------------  --------------  -----------------  -----------------  ----------------
275            37950           138.0              275                1               


In [260]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    ArtistId,
    Name,
    COUNT(*) AS occurrences
FROM artists
GROUP BY ArtistId, Name
HAVING COUNT(*) > 1;

In [261]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM artists
    GROUP BY ArtistId, Name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [262]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(ArtistId) FROM artists

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(ArtistId) FROM artists)
)

SELECT id AS missing_artist_id
FROM expected_ids
WHERE id NOT IN (
    SELECT ArtistId FROM artists
);

### 2. Albums

In [263]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from albums
limit 5

AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       


In [264]:
# Select specific columns
%%script sqlite3 --column --header chinook.db
SELECT
    AlbumId,
    Title
FROM albums
LIMIT 5;

AlbumId  Title                                
-------  -------------------------------------
1        For Those About To Rock We Salute You
2        Balls to the Wall                    
3        Restless and Wild                    
4        Let There Be Rock                    
5        Big Ones                             


In [265]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
PRAGMA table_info(albums);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    AlbumId   INTEGER        1                    1 
1    Title     NVARCHAR(160)  1                    0 
2    ArtistId  INTEGER        1                    0 


In [266]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('albums')
WHERE pk > 0;

name     type     pk
-------  -------  --
AlbumId  INTEGER  1 


In [267]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('albums');

id  seq  table    from      to        on_update  on_delete  match
--  ---  -------  --------  --------  ---------  ---------  -----
0   0    artists  ArtistId  ArtistId  NO ACTION  NO ACTION  NONE 


In [268]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('albums')

total_columns
-------------
3            


In [269]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_albums
from albums

total_albums
------------
347         


In [270]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_titles
FROM albums
WHERE title IS NULL;

null_titles
-----------
0          


In [271]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL THEN 1 ELSE 0 END) AS null_album_ids,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_titles
FROM albums;

total_rows  null_album_ids  null_titles
----------  --------------  -----------
347         0               0          


In [272]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL OR title IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM albums;

total_rows  rows_with_nulls
----------  ---------------
347         0              


In [273]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN albumId IS NULL THEN 1 ELSE 0 END) AS null_album_ids,
    SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS null_titles,
    SUM(
        CASE
            WHEN albumId IS NULL OR title IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM albums;

total_rows  null_album_ids  null_titles  rows_with_nulls
----------  --------------  -----------  ---------------
347         0               0            0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [274]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE title LIKE 'A%';

AlbumId  Title                                                    
-------  ---------------------------------------------------------
10       Audioslave                                               
14       Alcohol Fueled Brewtality Live! [Disc 1]                 
15       Alcohol Fueled Brewtality Live! [Disc 2]                 
24       Afrociberdelia                                           
26       Acústico MTV [Live]                                      
29       Axé Bahia 2001                                           
74       Album Of The Year                                        
75       Angel Dust                                               
85       As Canções de Eu Tu Eles                                 
89       American Idiot                                           
90       Appetite for Destruction                                 
94       A Matter of Life and Death                               
95       A Real Dead One                                      

In [275]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE title LIKE 'A_%';

AlbumId  Title                                                    
-------  ---------------------------------------------------------
10       Audioslave                                               
14       Alcohol Fueled Brewtality Live! [Disc 1]                 
15       Alcohol Fueled Brewtality Live! [Disc 2]                 
24       Afrociberdelia                                           
26       Acústico MTV [Live]                                      
29       Axé Bahia 2001                                           
74       Album Of The Year                                        
75       Angel Dust                                               
85       As Canções de Eu Tu Eles                                 
89       American Idiot                                           
90       Appetite for Destruction                                 
94       A Matter of Life and Death                               
95       A Real Dead One                                      

In [276]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE albumId BETWEEN 1 AND 50
  AND title LIKE '%Z%';

AlbumId  Title                 
-------  ----------------------
22       Sozinho Remix Ao Vivo 
33       Chill: Brazil (Disc 1)
34       Chill: Brazil (Disc 2)


In [277]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT albumId, title
FROM albums
WHERE albumId = 1
   OR albumId = 2
   OR albumId = 3;

AlbumId  Title                                
-------  -------------------------------------
1        For Those About To Rock We Salute You
2        Balls to the Wall                    
3        Restless and Wild                    


In [278]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM albums
LIMIT 5;

AlbumId  Title                                  ArtistId
-------  -------------------------------------  --------
1        For Those About To Rock We Salute You  1       
2        Balls to the Wall                      2       
3        Restless and Wild                      2       
4        Let There Be Rock                      1       
5        Big Ones                               3       


In [279]:
# Count unique album titles
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT title) AS unique_album_titles
FROM albums;

unique_album_titles
-------------------
347                


In [280]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    albumId,
    title
FROM albums
ORDER BY title ASC;

AlbumId  Title                                                                                          
-------  -----------------------------------------------------------------------------------------------
156      ...And Justice For All                                                                         
257      20th Century Masters - The Millennium Collection: The Best of Scorpions                        
296      A Copland Celebration, Vol. I                                                                  
94       A Matter of Life and Death                                                                     
95       A Real Dead One                                                                                
96       A Real Live One                                                                                
285      A Soprano Inspired                                                                             
139      A TempestadeTempestade Ou O Livro Dos Dias    

In [281]:
# GROUP BY
%%script sqlite3 --column --header chinook.db
SELECT
    title,
    COUNT(*) AS occurrences
FROM albums
GROUP BY title
ORDER BY occurrences DESC;

Title                                                                                            occurrences
-----------------------------------------------------------------------------------------------  -----------
[1997] Black Light Syndrome                                                                      1          
Zooropa                                                                                          1          
Worlds                                                                                           1          
Weill: The Seven Deadly Sins                                                                     1          
Warner 25 Anos                                                                                   1          
War                                                                                              1          
Walking Into Clarksdale                                                                          1          
Wagner: Favourite O

In [282]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT title, COUNT(*) AS occurrences
FROM albums
GROUP BY title
ORDER BY title
LIMIT 10;

Title                                                                    occurrences
-----------------------------------------------------------------------  -----------
...And Justice For All                                                   1          
20th Century Masters - The Millennium Collection: The Best of Scorpions  1          
A Copland Celebration, Vol. I                                            1          
A Matter of Life and Death                                               1          
A Real Dead One                                                          1          
A Real Live One                                                          1          
A Soprano Inspired                                                       1          
A TempestadeTempestade Ou O Livro Dos Dias                               1          
A-Sides                                                                  1          
Ace Of Spades                                                    

In [283]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_albums,
    SUM(albumId) AS sum_album_ids,
    AVG(albumId) AS average_album_id,
    MAX(albumId) AS highest_album_id,
    MIN(albumId) AS lowest_album_id
FROM albums;

total_albums  sum_album_ids  average_album_id  highest_album_id  lowest_album_id
------------  -------------  ----------------  ----------------  ---------------
347           60378          174.0             347               1              


In [284]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    albumId,
    title,
    COUNT(*) AS occurrences
FROM albums
GROUP BY albumId, title
HAVING COUNT(*) > 1;

In [285]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM albums
    GROUP BY albumId, title
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [286]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(albumId) FROM albums

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(albumId) FROM albums)
)

SELECT id AS missing_album_id
FROM expected_ids
WHERE id NOT IN (
    SELECT albumId FROM albums
);

In [287]:
# Foreign key check
%%script sqlite3 --column --header chinook.db
SELECT
    AlbumId,
    title,
    ArtistId
FROM albums
WHERE NOT EXISTS (
    SELECT 1
    FROM artists
    WHERE artists.ArtistId = albums.ArtistId
);

### 3. Tracks

In [288]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from tracks
limit 5

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

In [289]:
# Select specific columns
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name
FROM tracks
LIMIT 5;

TrackId  Name                                   
-------  ---------------------------------------
1        For Those About To Rock (We Salute You)
2        Balls to the Wall                      
3        Fast As a Shark                        
4        Restless and Wild                      
5        Princess of the Dawn                   


In [290]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
PRAGMA table_info(tracks);

cid  name          type           notnull  dflt_value  pk
---  ------------  -------------  -------  ----------  --
0    TrackId       INTEGER        1                    1 
1    Name          NVARCHAR(200)  1                    0 
2    AlbumId       INTEGER        0                    0 
3    MediaTypeId   INTEGER        1                    0 
4    GenreId       INTEGER        0                    0 
5    Composer      NVARCHAR(220)  0                    0 
6    Milliseconds  INTEGER        1                    0 
7    Bytes         INTEGER        0                    0 
8    UnitPrice     NUMERIC(10,2)  1                    0 


In [291]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('tracks')
WHERE pk > 0;

name     type     pk
-------  -------  --
TrackId  INTEGER  1 


In [292]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('tracks');

id  seq  table        from         to           on_update  on_delete  match
--  ---  -----------  -----------  -----------  ---------  ---------  -----
0   0    media_types  MediaTypeId  MediaTypeId  NO ACTION  NO ACTION  NONE 
1   0    genres       GenreId      GenreId      NO ACTION  NO ACTION  NONE 
2   0    albums       AlbumId      AlbumId      NO ACTION  NO ACTION  NONE 


In [293]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('tracks')

total_columns
-------------
9            


In [294]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_tracks
from tracks

total_tracks
------------
3503        


In [295]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM tracks
WHERE name IS NULL;

null_names
----------
0         


In [296]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL THEN 1 ELSE 0 END) AS null_track_ids,
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM tracks;

total_rows  null_track_ids  null_names
----------  --------------  ----------
3503        0               0         


In [297]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL OR name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM tracks;

total_rows  rows_with_nulls
----------  ---------------
3503        0              


In [298]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN trackId IS NULL THEN 1 ELSE 0 END) AS null_track_ids,
    SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN trackId IS NULL OR name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM tracks;

total_rows  null_track_ids  null_names  rows_with_nulls
----------  --------------  ----------  ---------------
3503        0               0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [299]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE name LIKE 'A%';

TrackId  Name                                                            
-------  ----------------------------------------------------------------
30       Amazing                                                         
36       Angel                                                           
38       All I Really Want                                               
72       Angela                                                          
134      All For You                                                     
139      A.N.D.R.O.T.A.Z.                                                
170      A Statistic                                                     
220      A Luz De Tieta                                                  
221      Atrás Da Verd-E-Rosa Só Não Vai Quem Já Morreu                  
231      Atras Da Porta                                                  
235      Apesar De Você                                                  
236      A Banda                      

In [300]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE name LIKE 'A_%';

TrackId  Name                                                            
-------  ----------------------------------------------------------------
30       Amazing                                                         
36       Angel                                                           
38       All I Really Want                                               
72       Angela                                                          
134      All For You                                                     
139      A.N.D.R.O.T.A.Z.                                                
170      A Statistic                                                     
220      A Luz De Tieta                                                  
221      Atrás Da Verd-E-Rosa Só Não Vai Quem Já Morreu                  
231      Atras Da Porta                                                  
235      Apesar De Você                                                  
236      A Banda                      

In [301]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE trackId BETWEEN 1 AND 50
  AND name LIKE '%Z%';

TrackId  Name   
-------  -------
30       Amazing
34       Crazy  


In [302]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT trackId, name
FROM tracks
WHERE trackId = 1
   OR trackId = 2
   OR trackId = 3;

TrackId  Name                                   
-------  ---------------------------------------
1        For Those About To Rock (We Salute You)
2        Balls to the Wall                      
3        Fast As a Shark                        


In [303]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM tracks
LIMIT 5;

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

In [304]:
# Count unique track names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT name) AS unique_track_names
FROM tracks;

unique_track_names
------------------
3257              


In [305]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name
FROM tracks
ORDER BY name ASC;

TrackId  Name                                                                                                                       
-------  ---------------------------------------------------------------------------------------------------------------------------
3027     "40"                                                                                                                       
2918     "?"                                                                                                                        
3412     "Eine Kleine Nachtmusik" Serenade In G, K. 525: I. Allegro                                                                 
109      #1 Zero                                                                                                                    
3254     #9 Dream                                                                                                                   
602      'Round Midnight                                             

In [306]:
# GROUP BY
%%script sqlite3 --column --header chinook.db
SELECT
    name,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY name
ORDER BY TrackCount DESC;

Name                                                                                                                         TrackCount
---------------------------------------------------------------------------------------------------------------------------  ----------
Wrathchild                                                                                                                   5         
The Trooper                                                                                                                  5         
The Number Of The Beast                                                                                                      5         
Iron Maiden                                                                                                                  5         
Hallowed Be Thy Name                                                                                                         5         
2 Minutes To Midnight                           

In [307]:
%%script sqlite3 --column --header chinook.db

SELECT
    AlbumId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY AlbumId
ORDER BY TrackCount DESC;

AlbumId  TrackCount
-------  ----------
141      57        
23       34        
73       30        
229      26        
230      25        
251      25        
83       24        
231      24        
253      24        
24       23        
228      23        
255      23        
51       22        
224      22        
250      22        
39       21        
167      21        
37       20        
54       20        
55       20        
115      20        
221      20        
227      19        
248      19        
258      19        
21       18        
72       18        
102      18        
145      18        
146      18        
202      18        
211      18        
213      18        
18       17        
26       17        
33       17        
34       17        
36       17        
120      17        
162      17        
163      17        
185      17        
193      17        
203      17        
237      17        
243      17        
259      17        
261      17        


In [308]:
%%script sqlite3 --column --header chinook.db

SELECT
    GenreId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY GenreId
ORDER BY TrackCount DESC;

GenreId  TrackCount
-------  ----------
1        1297      
7        579       
3        374       
4        332       
2        130       
19       93        
6        81        
24       74        
21       64        
14       61        
8        58        
9        48        
10       43        
23       40        
17       35        
15       30        
13       28        
16       28        
20       26        
12       24        
22       17        
11       15        
18       13        
5        12        
25       1         


In [309]:
%%script sqlite3 --column --header chinook.db

SELECT
    MediaTypeId,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY MediaTypeId
ORDER BY TrackCount DESC;

MediaTypeId  TrackCount
-----------  ----------
1            3034      
2            237       
3            214       
5            11        
4            7         


In [310]:
%%script sqlite3 --column --header chinook.db

SELECT
    Composer,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY Composer
ORDER BY TrackCount DESC
LIMIT 15;

Composer                                        TrackCount
----------------------------------------------  ----------
                                                978       
Steve Harris                                    80        
U2                                              44        
Jagger/Richards                                 35        
Billy Corgan                                    31        
Kurt Cobain                                     26        
Bill Berry-Peter Buck-Mike Mills-Michael Stipe  25        
The Tea Party                                   24        
Miles Davis                                     23        
Gilberto Gil                                    23        
Chris Cornell                                   23        
Chico Science                                   23        
Titãs                                           22        
Renato Russo                                    20        
J.C. Fogerty                                    20      

In [311]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT name, COUNT(*) AS occurrences
FROM tracks
GROUP BY name
ORDER BY name
LIMIT 10;

Name                                                        occurrences
----------------------------------------------------------  -----------
"40"                                                        1          
"?"                                                         1          
"Eine Kleine Nachtmusik" Serenade In G, K. 525: I. Allegro  1          
#1 Zero                                                     1          
#9 Dream                                                    1          
'Round Midnight                                             1          
(Anesthesia) Pulling Teeth                                  1          
(Da Le) Yaleo                                               1          
(I Can't Help) Falling In Love With You                     1          
(Oh) Pretty Woman                                           1          


In [312]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_tracks,
    SUM(trackId) AS sum_track_ids,
    AVG(trackId) AS average_track_id,
    MAX(trackId) AS highest_track_id,
    MIN(trackId) AS lowest_track_id
FROM tracks;

total_tracks  sum_track_ids  average_track_id  highest_track_id  lowest_track_id
------------  -------------  ----------------  ----------------  ---------------
3503          6137256        1752.0            3503              1              


In [313]:
# Check out float features
%%script sqlite3 --column --header chinook.db

SELECT
    MIN(Milliseconds) AS ShortestTrack,
    MAX(Milliseconds) AS LongestTrack,
    ROUND(AVG(Milliseconds), 2) AS AverageMilliseconds,
    SUM(Milliseconds) AS TotalMilliseconds
FROM tracks;

SELECT
    MIN(Bytes) AS SmallestFile,
    MAX(Bytes) AS LargestFile,
    ROUND(AVG(Bytes), 2) AS AverageBytes
FROM tracks;

SELECT
    MIN(UnitPrice) AS LowestPrice,
    MAX(UnitPrice) AS HighestPrice,
    AVG(UnitPrice) AS AveragePrice
FROM tracks;

ShortestTrack  LongestTrack  AverageMilliseconds  TotalMilliseconds
-------------  ------------  -------------------  -----------------
1071           5286953       393599.21            1378778040       
SmallestFile  LargestFile  AverageBytes
------------  -----------  ------------
38747         1059546140   33510207.07 
LowestPrice  HighestPrice  AveragePrice    
-----------  ------------  ----------------
0.99         1.99          1.05080502426492


In [314]:
# Duration in minutes
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    ROUND(Milliseconds / 60000.0, 2) AS DurationMinutes
FROM tracks
ORDER BY Milliseconds DESC
LIMIT 10;

Name                         DurationMinutes
---------------------------  ---------------
Occupation / Precipice       88.12          
Through a Looking Glass      84.81          
Greetings from Earth, Pt. 1  49.34          
The Man With Nine Lives      49.28          
Battlestar Galactica, Pt. 2  49.27          
Battlestar Galactica, Pt. 1  49.21          
Murder On the Rising Star    48.93          
Battlestar Galactica, Pt. 3  48.8           
Take the Celestra            48.79          
Fire In Space                48.78          


In [315]:
# Frequency of track by price
%%script sqlite3 --column --header chinook.db

SELECT
    UnitPrice,
    COUNT(*) AS TrackCount
FROM tracks
GROUP BY UnitPrice
ORDER BY UnitPrice;

UnitPrice  TrackCount
---------  ----------
0.99       3290      
1.99       213       


In [316]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    trackId,
    name,
    COUNT(*) AS occurrences
FROM tracks
GROUP BY trackId, name
HAVING COUNT(*) > 1;

In [317]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM tracks
    GROUP BY trackId, name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [318]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(trackId) FROM tracks

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(trackId) FROM tracks)
)

SELECT id AS missing_track_id
FROM expected_ids
WHERE id NOT IN (
    SELECT trackId FROM tracks
);

In [319]:
# Foreign key check: AlbumId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    AlbumId
FROM tracks
WHERE AlbumId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM albums
    WHERE albums.AlbumId = tracks.AlbumId
);

In [320]:
# Foreign key check: MediaTypeId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    MediaTypeId
FROM tracks
WHERE MediaTypeId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM media_types
    WHERE media_types.MediaTypeId = tracks.MediaTypeId
);

In [321]:
# Foreign key check: GenreId
%%script sqlite3 --column --header chinook.db

SELECT
    TrackId,
    Name,
    GenreId
FROM tracks
WHERE GenreId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM genres
    WHERE genres.GenreId = tracks.GenreId
);

In [322]:
# Alternative: use joins
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS InvalidAlbumReferences
FROM tracks t
LEFT JOIN albums a ON t.AlbumId = a.AlbumId
WHERE t.AlbumId IS NOT NULL
AND a.AlbumId IS NULL;

SELECT
    COUNT(*) AS InvalidMediaTypeReferences
FROM tracks t
LEFT JOIN media_types m ON t.MediaTypeId = m.MediaTypeId
WHERE t.MediaTypeId IS NOT NULL
AND m.MediaTypeId IS NULL;

SELECT
    COUNT(*) AS InvalidGenreReferences
FROM tracks t
LEFT JOIN genres g ON t.GenreId = g.GenreId
WHERE t.GenreId IS NOT NULL
AND g.GenreId IS NULL;

InvalidAlbumReferences
----------------------
0                     
InvalidMediaTypeReferences
--------------------------
0                         
InvalidGenreReferences
----------------------
0                     


### 4. Genres

In [323]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from genres
limit 5

GenreId  Name              
-------  ------------------
1        Rock              
2        Jazz              
3        Metal             
4        Alternative & Punk
5        Rock And Roll     


In [324]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    genreId,
    Name
FROM genres
LIMIT 10;

GenreId  Name              
-------  ------------------
1        Rock              
2        Jazz              
3        Metal             
4        Alternative & Punk
5        Rock And Roll     
6        Blues             
7        Latin             
8        Reggae            
9        Pop               
10       Soundtrack        


In [325]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('genres') -- must have 'genres', not genres
PRAGMA table_info(genres);

cid  name     type           notnull  dflt_value  pk
---  -------  -------------  -------  ----------  --
0    GenreId  INTEGER        1                    1 
1    Name     NVARCHAR(120)  0                    0 


In [326]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('genres')
WHERE pk > 0;

name     type     pk
-------  -------  --
GenreId  INTEGER  1 


In [327]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('genres');

In [328]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('genres')

total_columns
-------------
2            


In [329]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_genres
from genres

total_genres
------------
25          


#### Nulls:

In [330]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM genres
WHERE Name IS NULL;

null_names
----------
0         


In [331]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN genreId IS NULL THEN 1 ELSE 0 END) AS null_genre_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM genres;

total_rows  null_genre_ids  null_names
----------  --------------  ----------
25          0               0         


In [332]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN genreId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM genres;

total_rows  rows_with_nulls
----------  ---------------
25          0              


In [333]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN genreId IS NULL THEN 1 ELSE 0 END) AS null_genre_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN genreId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM genres;

total_rows  null_genre_ids  null_names  rows_with_nulls
----------  --------------  ----------  ---------------
25          0               0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [334]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT genreId, Name
FROM genres
WHERE Name LIKE 'A%';

GenreId  Name              
-------  ------------------
4        Alternative & Punk
23       Alternative       


In [335]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT genreId, Name
FROM genres
WHERE Name LIKE 'A_%';

GenreId  Name              
-------  ------------------
4        Alternative & Punk
23       Alternative       


In [336]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT genreId, Name
FROM genres
WHERE genreId BETWEEN 1 AND 50
  AND Name LIKE '%H%';

GenreId  Name       
-------  -----------
13       Heavy Metal
17       Hip Hop/Rap
19       TV Shows   


In [337]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT genreId, Name
FROM genres
WHERE genreId = 1
   OR genreId = 2
   OR genreId = 3;

GenreId  Name 
-------  -----
1        Rock 
2        Jazz 
3        Metal


In [338]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM genres
LIMIT 5;

GenreId  Name              
-------  ------------------
1        Rock              
2        Jazz              
3        Metal             
4        Alternative & Punk
5        Rock And Roll     


#### DISTINCT, GROUP BY, ORDER BY,

In [339]:
# Count unique genre names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_genre_names
FROM genres;

unique_genre_names
------------------
25                


In [340]:
# View unique names (categories) in genre table
%%script sqlite3 --column --header chinook.db

-- List distinct genre categories
SELECT DISTINCT Name
FROM genres
ORDER BY Name;

Name              
------------------
Alternative       
Alternative & Punk
Blues             
Bossa Nova        
Classical         
Comedy            
Drama             
Easy Listening    
Electronica/Dance 
Heavy Metal       
Hip Hop/Rap       
Jazz              
Latin             
Metal             
Opera             
Pop               
R&B/Soul          
Reggae            
Rock              
Rock And Roll     
Sci Fi & Fantasy  
Science Fiction   
Soundtrack        
TV Shows          
World             


In [341]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    genreId,
    Name
FROM genres
ORDER BY Name ASC;

GenreId  Name              
-------  ------------------
23       Alternative       
4        Alternative & Punk
6        Blues             
11       Bossa Nova        
24       Classical         
22       Comedy            
21       Drama             
12       Easy Listening    
15       Electronica/Dance 
13       Heavy Metal       
17       Hip Hop/Rap       
2        Jazz              
7        Latin             
3        Metal             
25       Opera             
9        Pop               
14       R&B/Soul          
8        Reggae            
1        Rock              
5        Rock And Roll     
20       Sci Fi & Fantasy  
18       Science Fiction   
10       Soundtrack        
19       TV Shows          
16       World             


In [342]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    COUNT(*) AS occurrences
FROM genres
GROUP BY Name
ORDER BY occurrences DESC;

Name                occurrences
------------------  -----------
World               1          
TV Shows            1          
Soundtrack          1          
Science Fiction     1          
Sci Fi & Fantasy    1          
Rock And Roll       1          
Rock                1          
Reggae              1          
R&B/Soul            1          
Pop                 1          
Opera               1          
Metal               1          
Latin               1          
Jazz                1          
Hip Hop/Rap         1          
Heavy Metal         1          
Electronica/Dance   1          
Easy Listening      1          
Drama               1          
Comedy              1          
Classical           1          
Bossa Nova          1          
Blues               1          
Alternative & Punk  1          
Alternative         1          


In [343]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM genres
GROUP BY Name
ORDER BY Name
LIMIT 10;

Name                occurrences
------------------  -----------
Alternative         1          
Alternative & Punk  1          
Blues               1          
Bossa Nova          1          
Classical           1          
Comedy              1          
Drama               1          
Easy Listening      1          
Electronica/Dance   1          
Heavy Metal         1          


In [344]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_genres,
    SUM(genreId) AS sum_genre_ids,
    AVG(genreId) AS average_genre_id,
    MAX(genreId) AS highest_genre_id,
    MIN(genreId) AS lowest_genre_id
FROM genres;

total_genres  sum_genre_ids  average_genre_id  highest_genre_id  lowest_genre_id
------------  -------------  ----------------  ----------------  ---------------
25            325            13.0              25                1              


In [345]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    Name,
    COUNT(*) AS occurrences
FROM genres
GROUP BY Name
HAVING COUNT(*) > 1;

In [346]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    genreId,
    Name,
    COUNT(*) AS occurrences
FROM genres
GROUP BY genreId, Name
HAVING COUNT(*) > 1;

In [347]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM genres
    GROUP BY genreId, Name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [348]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(genreId) FROM genres

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(genreId) FROM genres)
)

SELECT id AS missing_genre_id
FROM expected_ids
WHERE id NOT IN (
    SELECT genreId FROM genres
);

In [349]:
# Look for unused genres
%%script sqlite3 --column --header chinook.db

-- Find genres that are not referenced by any tracks
SELECT
    GenreId,
    Name
FROM genres
WHERE NOT EXISTS (
    SELECT 1
    FROM tracks
    WHERE tracks.GenreId = genres.GenreId
);

### 5. Media_Types

In [350]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from media_types
limit 5

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   
5            AAC audio file             


In [351]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    mediatypeId,
    Name
FROM media_types
LIMIT 10;

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   
5            AAC audio file             


In [352]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('media_types') -- must have 'media_types', not media_types
PRAGMA table_info(media_types);

cid  name         type           notnull  dflt_value  pk
---  -----------  -------------  -------  ----------  --
0    MediaTypeId  INTEGER        1                    1 
1    Name         NVARCHAR(120)  0                    0 


In [353]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('media_types')
WHERE pk > 0;

name         type     pk
-----------  -------  --
MediaTypeId  INTEGER  1 


In [354]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('media_types');

In [355]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('media_types')

total_columns
-------------
2            


In [356]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_media_types
from media_types

total_media_types
-----------------
5                


#### Nulls:

In [357]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM media_types
WHERE Name IS NULL;

null_names
----------
0         


In [358]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN mediatypeId IS NULL THEN 1 ELSE 0 END) AS null_media_type_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM media_types;

total_rows  null_media_type_ids  null_names
----------  -------------------  ----------
5           0                    0         


In [359]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN mediatypeId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM media_types;

total_rows  rows_with_nulls
----------  ---------------
5           0              


In [360]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN mediatypeId IS NULL THEN 1 ELSE 0 END) AS null_media_type_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN mediatypeId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM media_types;

total_rows  null_media_type_ids  null_names  rows_with_nulls
----------  -------------------  ----------  ---------------
5           0                    0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [361]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT mediatypeId, Name
FROM media_types
WHERE Name LIKE 'A%';

MediaTypeId  Name          
-----------  --------------
5            AAC audio file


In [362]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT mediatypeId, Name
FROM media_types
WHERE Name LIKE 'A_%';

MediaTypeId  Name          
-----------  --------------
5            AAC audio file


In [363]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT mediatypeId, Name
FROM media_types
WHERE mediatypeId BETWEEN 1 AND 50
  AND Name LIKE '%H%';

MediaTypeId  Name                    
-----------  ------------------------
4            Purchased AAC audio file


In [364]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT mediatypeId, Name
FROM media_types
WHERE mediatypeId = 1
   OR mediatypeId = 2
   OR mediatypeId = 3;

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file


In [365]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM media_types
LIMIT 5;

MediaTypeId  Name                       
-----------  ---------------------------
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   
5            AAC audio file             


#### DISTINCT, GROUP BY, ORDER BY

In [366]:
# Count unique media_type names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_media_type_names
FROM media_types;

unique_media_type_names
-----------------------
5                      


In [367]:
# View unique names (categories) in media_type table
%%script sqlite3 --column --header chinook.db

-- List distinct media_type categories
SELECT DISTINCT Name
FROM media_types
ORDER BY Name;

Name                       
---------------------------
AAC audio file             
MPEG audio file            
Protected AAC audio file   
Protected MPEG-4 video file
Purchased AAC audio file   


In [368]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    mediatypeId,
    Name
FROM media_types
ORDER BY Name ASC;

MediaTypeId  Name                       
-----------  ---------------------------
5            AAC audio file             
1            MPEG audio file            
2            Protected AAC audio file   
3            Protected MPEG-4 video file
4            Purchased AAC audio file   


In [369]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    COUNT(*) AS occurrences
FROM media_types
GROUP BY Name
ORDER BY occurrences DESC;

Name                         occurrences
---------------------------  -----------
Purchased AAC audio file     1          
Protected MPEG-4 video file  1          
Protected AAC audio file     1          
MPEG audio file              1          
AAC audio file               1          


In [370]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM media_types
GROUP BY Name
ORDER BY Name
LIMIT 10;

Name                         occurrences
---------------------------  -----------
AAC audio file               1          
MPEG audio file              1          
Protected AAC audio file     1          
Protected MPEG-4 video file  1          
Purchased AAC audio file     1          


#### MAX, MIN, SUM, AVG, COUNT

In [371]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_media_types,
    SUM(mediatypeId) AS sum_media_type_ids,
    AVG(mediatypeId) AS average_media_type_id,
    MAX(mediatypeId) AS highest_media_type_id,
    MIN(mediatypeId) AS lowest_media_type_id
FROM media_types;

total_media_types  sum_media_type_ids  average_media_type_id  highest_media_type_id  lowest_media_type_id
-----------------  ------------------  ---------------------  ---------------------  --------------------
5                  15                  3.0                    5                      1                   


In [372]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    Name,
    COUNT(*) AS occurrences
FROM media_types
GROUP BY Name
HAVING COUNT(*) > 1;

In [373]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    mediatypeId,
    Name,
    COUNT(*) AS occurrences
FROM media_types
GROUP BY mediatypeId, Name
HAVING COUNT(*) > 1;

In [374]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM media_types
    GROUP BY mediatypeId, Name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [375]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(mediatypeId) FROM media_types

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(mediatypeId) FROM media_types)
)

SELECT id AS missing_media_type_id
FROM expected_ids
WHERE id NOT IN (
    SELECT mediatypeId FROM media_types
);

In [376]:
# Look for unused media_types
%%script sqlite3 --column --header chinook.db

-- Find media_types that are not referenced by any tracks
SELECT
    mediatypeId,
    Name
FROM media_types
WHERE NOT EXISTS (
    SELECT 1
    FROM tracks
    WHERE tracks.mediatypeId = media_types.mediatypeId
);

In [377]:
# Check foreign-key references
%%script sqlite3 --column --header chinook.db

-- Check for invalid MediaTypeId references in tracks
SELECT
    TrackId,
    MediaTypeId
FROM tracks
WHERE MediaTypeId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM media_types
    WHERE media_types.MediaTypeId = tracks.MediaTypeId
);

### 6. Playlists

In [378]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from playlists
limit 5

PlaylistId  Name      
----------  ----------
1           Music     
2           Movies    
3           TV Shows  
4           Audiobooks
5           90’s Music


In [379]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    playlistId,
    Name
FROM playlists
LIMIT 10;

PlaylistId  Name        
----------  ------------
1           Music       
2           Movies      
3           TV Shows    
4           Audiobooks  
5           90’s Music  
6           Audiobooks  
7           Movies      
8           Music       
9           Music Videos
10          TV Shows    


In [380]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('playlists') -- must have 'playlists', not playlists
PRAGMA table_info(playlists);

cid  name        type           notnull  dflt_value  pk
---  ----------  -------------  -------  ----------  --
0    PlaylistId  INTEGER        1                    1 
1    Name        NVARCHAR(120)  0                    0 


In [381]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('playlists')
WHERE pk > 0;

name        type     pk
----------  -------  --
PlaylistId  INTEGER  1 


In [382]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('playlists');

In [383]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('playlists')

total_columns
-------------
2            


In [384]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_playlists
from playlists

total_playlists
---------------
18             


#### Nulls:

In [385]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM playlists
WHERE Name IS NULL;

null_names
----------
0         


In [386]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistId IS NULL THEN 1 ELSE 0 END) AS null_media_type_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names
FROM playlists;

total_rows  null_media_type_ids  null_names
----------  -------------------  ----------
18          0                    0         


In [387]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistId IS NULL OR Name IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM playlists;

total_rows  rows_with_nulls
----------  ---------------
18          0              


In [388]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistId IS NULL THEN 1 ELSE 0 END) AS null_media_type_ids,
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN playlistId IS NULL OR Name IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM playlists;

total_rows  null_media_type_ids  null_names  rows_with_nulls
----------  -------------------  ----------  ---------------
18          0                    0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [389]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT playlistId, Name
FROM playlists
WHERE Name LIKE 'A%';

PlaylistId  Name      
----------  ----------
4           Audiobooks
6           Audiobooks


In [390]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT playlistId, Name
FROM playlists
WHERE Name LIKE 'A_%';

PlaylistId  Name      
----------  ----------
4           Audiobooks
6           Audiobooks


In [391]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT playlistId, Name
FROM playlists
WHERE playlistId BETWEEN 1 AND 50
  AND Name LIKE '%H%';

PlaylistId  Name                      
----------  --------------------------
3           TV Shows                  
10          TV Shows                  
15          Classical 101 - The Basics
17          Heavy Metal Classic       
18          On-The-Go 1               


In [392]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT playlistId, Name
FROM playlists
WHERE playlistId = 1
   OR playlistId = 2
   OR playlistId = 3;

PlaylistId  Name    
----------  --------
1           Music   
2           Movies  
3           TV Shows


In [393]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM playlists
LIMIT 5;

PlaylistId  Name      
----------  ----------
1           Music     
2           Movies    
3           TV Shows  
4           Audiobooks
5           90’s Music


#### DISTINCT, GROUP BY, ORDER BY

In [394]:
# Count unique media_type names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT Name) AS unique_media_type_names
FROM playlists;

unique_media_type_names
-----------------------
14                     


In [395]:
# View unique names (categories) in media_type table
%%script sqlite3 --column --header chinook.db

-- List distinct playlist categories
SELECT DISTINCT Name
FROM playlists
ORDER BY Name;

Name                      
--------------------------
90’s Music                
Audiobooks                
Brazilian Music           
Classical                 
Classical 101 - Deep Cuts 
Classical 101 - Next Steps
Classical 101 - The Basics
Grunge                    
Heavy Metal Classic       
Movies                    
Music                     
Music Videos              
On-The-Go 1               
TV Shows                  


In [396]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    playlistId,
    Name
FROM playlists
ORDER BY Name ASC;

PlaylistId  Name                      
----------  --------------------------
5           90’s Music                
4           Audiobooks                
6           Audiobooks                
11          Brazilian Music           
12          Classical                 
13          Classical 101 - Deep Cuts 
14          Classical 101 - Next Steps
15          Classical 101 - The Basics
16          Grunge                    
17          Heavy Metal Classic       
2           Movies                    
7           Movies                    
1           Music                     
8           Music                     
9           Music Videos              
18          On-The-Go 1               
3           TV Shows                  
10          TV Shows                  


In [397]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    Name,
    COUNT(*) AS PlaylistCount
FROM playlists
GROUP BY Name
ORDER BY PlaylistCount DESC;

Name                        PlaylistCount
--------------------------  -------------
TV Shows                    2            
Music                       2            
Movies                      2            
Audiobooks                  2            
On-The-Go 1                 1            
Music Videos                1            
Heavy Metal Classic         1            
Grunge                      1            
Classical 101 - The Basics  1            
Classical 101 - Next Steps  1            
Classical 101 - Deep Cuts   1            
Classical                   1            
Brazilian Music             1            
90’s Music                  1            


In [398]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT Name, COUNT(*) AS occurrences
FROM playlists
GROUP BY Name
ORDER BY Name
LIMIT 10;

Name                        occurrences
--------------------------  -----------
90’s Music                  1          
Audiobooks                  2          
Brazilian Music             1          
Classical                   1          
Classical 101 - Deep Cuts   1          
Classical 101 - Next Steps  1          
Classical 101 - The Basics  1          
Grunge                      1          
Heavy Metal Classic         1          
Movies                      2          


#### MAX, MIN, SUM, AVG, COUNT

In [399]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_playlists,
    SUM(playlistId) AS sum_playlist_ids,
    AVG(playlistId) AS average_playlist_id,
    MAX(playlistId) AS highest_playlist_id,
    MIN(playlistId) AS lowest_playlist_id
FROM playlists;

total_playlists  sum_playlist_ids  average_playlist_id  highest_playlist_id  lowest_playlist_id
---------------  ----------------  -------------------  -------------------  ------------------
18               171               9.5                  18                   1                 


In [400]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    Name,
    COUNT(*) AS occurrences
FROM playlists
GROUP BY Name
HAVING COUNT(*) > 1;

Name        occurrences
----------  -----------
Audiobooks  2          
Movies      2          
Music       2          
TV Shows    2          


In [401]:
# Identify duplicate playlist records
%%script sqlite3 --column --header chinook.db
SELECT
    playlistId,
    Name,
    COUNT(*) AS occurrences
FROM playlists
GROUP BY playlistId, Name
HAVING COUNT(*) > 1;

In [402]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM playlists
    GROUP BY playlistId, Name
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [403]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(playlistId) FROM playlists

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(playlistId) FROM playlists)
)

SELECT id AS missing_media_type_id
FROM expected_ids
WHERE id NOT IN (
    SELECT playlistId FROM playlists
);

In [404]:
# Look for unused playlists
%%script sqlite3 --column --header chinook.db

-- Find playlists that are not referenced by any tracks
SELECT
    playlistId,
    Name
FROM playlists
WHERE NOT EXISTS (
    SELECT 1
    FROM playlist_track
    WHERE playlist_track.playlistId = playlists.playlistId
);

PlaylistId  Name      
----------  ----------
2           Movies    
4           Audiobooks
6           Audiobooks
7           Movies    


In [405]:
# Check foreign-key references
%%script sqlite3 --column --header chinook.db

-- Check for invalid playlistId references in tracks
SELECT
    TrackId,
    playlistId
FROM playlist_track
WHERE playlistId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM playlists
    WHERE playlists.playlistId = playlist_track.playlistId
);

### 7. Playlist_Track

In [406]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from playlist_track
limit 5

PlaylistId  TrackId
----------  -------
1           3402   
1           3389   
1           3390   
1           3391   
1           3392   


In [407]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    PlaylistId,
    TrackId
FROM playlist_track
LIMIT 10;

PlaylistId  TrackId
----------  -------
1           3402   
1           3389   
1           3390   
1           3391   
1           3392   
1           3393   
1           3394   
1           3395   
1           3396   
1           3397   


In [408]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('playlist_track') -- must have 'playlist_track', not playlist_track
PRAGMA table_info(playlist_track);

cid  name        type     notnull  dflt_value  pk
---  ----------  -------  -------  ----------  --
0    PlaylistId  INTEGER  1                    1 
1    TrackId     INTEGER  1                    2 


In [409]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('playlist_track')
WHERE pk > 0;

name        type     pk
----------  -------  --
PlaylistId  INTEGER  1 
TrackId     INTEGER  2 


In [410]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('playlist_track');

id  seq  table      from        to          on_update  on_delete  match
--  ---  ---------  ----------  ----------  ---------  ---------  -----
0   0    tracks     TrackId     TrackId     NO ACTION  NO ACTION  NONE 
1   0    playlists  PlaylistId  PlaylistId  NO ACTION  NO ACTION  NONE 


In [411]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('playlist_track')

total_columns
-------------
2            


In [412]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_playlist_track
from playlist_track

total_playlist_track
--------------------
8715                


In [413]:
# Count rows where trackid is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_track_ids
FROM playlist_track
WHERE trackid IS NULL;

null_track_ids
--------------
0             


In [414]:
# Count rows where trackid is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_playlist_ids
FROM playlist_track
WHERE playlistid IS NULL;

null_playlist_ids
-----------------
0                


In [415]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistid IS NULL THEN 1 ELSE 0 END) AS null_playlistids,
    SUM(CASE WHEN trackid IS NULL THEN 1 ELSE 0 END) AS null_trackids
FROM playlist_track;

total_rows  null_playlistids  null_trackids
----------  ----------------  -------------
8715        0                 0            


In [416]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistid IS NULL OR trackid IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM playlist_track;

total_rows  rows_with_nulls
----------  ---------------
8715        0              


In [417]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN playlistid IS NULL THEN 1 ELSE 0 END) AS null_playlistids,
    SUM(CASE WHEN trackid IS NULL THEN 1 ELSE 0 END) AS null_trackids,
    SUM(
        CASE
            WHEN playlistid IS NULL OR trackid IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM playlist_track;

total_rows  null_playlistids  null_trackids  rows_with_nulls
----------  ----------------  -------------  ---------------
8715        0                 0              0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [418]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT playlistid, trackid
FROM playlist_track
WHERE trackid LIKE 'A%';

In [419]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT playlistid, trackid
FROM playlist_track
WHERE trackid LIKE 'A_%';

In [420]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT playlistid, trackid
FROM playlist_track
WHERE playlistid BETWEEN 1 AND 50
  AND trackid LIKE '%H%';

In [421]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT playlistid, trackid
FROM playlist_track
WHERE playlistid = 1
   OR playlistid = 2
   OR playlistid = 3;

PlaylistId  TrackId
----------  -------
1           1      
1           2      
1           3      
1           4      
1           5      
1           6      
1           7      
1           8      
1           9      
1           10     
1           11     
1           12     
1           13     
1           14     
1           15     
1           16     
1           17     
1           18     
1           19     
1           20     
1           21     
1           22     
1           23     
1           24     
1           25     
1           26     
1           27     
1           28     
1           29     
1           30     
1           31     
1           32     
1           33     
1           34     
1           35     
1           36     
1           37     
1           38     
1           39     
1           40     
1           41     
1           42     
1           43     
1           44     
1           45     
1           46     
1           47     
1           48     


In [422]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM playlist_track
LIMIT 5;

PlaylistId  TrackId
----------  -------
1           3402   
1           3389   
1           3390   
1           3391   
1           3392   


In [423]:
# Count unique artist trackids
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT trackid)
FROM playlist_track;

COUNT(DISTINCT trackid)
-----------------------
3503                   


In [424]:
# Count unique artist trackids
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT playlistid)
FROM playlist_track;

COUNT(DISTINCT playlistid)
--------------------------
14                        


In [425]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    playlistid,
    trackid
FROM playlist_track
ORDER BY trackid ASC;

PlaylistId  TrackId
----------  -------
1           1      
8           1      
17          1      
1           2      
8           2      
17          2      
1           3      
5           3      
8           3      
17          3      
1           4      
5           4      
8           4      
17          4      
1           5      
5           5      
8           5      
17          5      
1           6      
8           6      
1           7      
8           7      
1           8      
8           8      
1           9      
8           9      
1           10     
8           10     
1           11     
8           11     
1           12     
8           12     
1           13     
8           13     
1           14     
8           14     
1           15     
8           15     
1           16     
8           16     
1           17     
8           17     
1           18     
8           18     
1           19     
8           19     
1           20     
8           20     


In [426]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    trackid,
    COUNT(*) AS occurrences
FROM playlist_track
GROUP BY trackid
ORDER BY occurrences DESC;

TrackId  occurrences
-------  -----------
3403     5          
3404     5          
3408     5          
3409     5          
3410     5          
3411     5          
3415     5          
3416     5          
3417     5          
3418     5          
3421     5          
3423     5          
3424     5          
3426     5          
3430     5          
3432     5          
3434     5          
3435     5          
3436     5          
3437     5          
3438     5          
3439     5          
3442     5          
3444     5          
3445     5          
3446     5          
3447     5          
3448     5          
3449     5          
3451     5          
3454     5          
3481     5          
3482     5          
3485     5          
3489     5          
3490     5          
3492     5          
3493     5          
3498     5          
3499     5          
3503     5          
3        4          
4        4          
5        4          
52       4          
215      4   

In [427]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT trackid, COUNT(*) AS occurrences
FROM playlist_track
GROUP BY trackid
ORDER BY trackid
LIMIT 10;

TrackId  occurrences
-------  -----------
1        3          
2        3          
3        4          
4        4          
5        4          
6        2          
7        2          
8        2          
9        2          
10       2          


In [428]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_playlist_track,
    SUM(playlistid) AS sum_artist_ids,
    AVG(playlistid) AS average_artist_id,
    MAX(playlistid) AS highest_artist_id,
    MIN(playlistid) AS lowest_artist_id
FROM playlist_track;

total_playlist_track  sum_artist_ids  average_artist_id  highest_artist_id  lowest_artist_id
--------------------  --------------  -----------------  -----------------  ----------------
8715                  42852           4.91703958691911   18                 1               


In [429]:
# Identify duplicate playlist-track combos
%%script sqlite3 --column --header chinook.db
SELECT
    playlistid,
    trackid,
    COUNT(*) AS occurrences
FROM playlist_track
GROUP BY playlistid, trackid
HAVING COUNT(*) > 1;

In [430]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM playlist_track
    GROUP BY playlistid, trackid
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [431]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(playlistid) FROM playlist_track

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(playlistid) FROM playlist_track)
)

SELECT id AS missing_artist_id
FROM expected_ids
WHERE id NOT IN (
    SELECT playlistid FROM playlist_track
);

missing_artist_id
-----------------
2                
4                
6                
7                


In [432]:
# Playlist frequency
%%script sqlite3 --column --header chinook.db

-- Count tracks associated with each playlist
SELECT
    PlaylistId,
    COUNT(*) AS TrackCount
FROM playlist_track
GROUP BY PlaylistId
ORDER BY TrackCount DESC;

PlaylistId  TrackCount
----------  ----------
1           3290      
8           3290      
5           1477      
3           213       
10          213       
12          75        
11          39        
17          26        
13          25        
14          25        
15          25        
16          15        
9           1         
18          1         


In [433]:
# Track frequency
%%script sqlite3 --column --header chinook.db

-- Count playlists associated with each track
SELECT
    TrackId,
    COUNT(*) AS PlaylistCount
FROM playlist_track
GROUP BY TrackId
ORDER BY PlaylistCount DESC
LIMIT 10;

TrackId  PlaylistCount
-------  -------------
3403     5            
3404     5            
3408     5            
3409     5            
3410     5            
3411     5            
3415     5            
3416     5            
3417     5            
3418     5            


In [434]:
# Validate TrackId references
%%script sqlite3 --column --header chinook.db

-- Identify invalid TrackId references
SELECT
    PlaylistId,
    TrackId
FROM playlist_track
WHERE TrackId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM tracks
    WHERE tracks.TrackId = playlist_track.TrackId
);

In [435]:
# Validate PlaylistId references
%%script sqlite3 --column --header chinook.db

-- Identify invalid PlaylistId references
SELECT
    PlaylistId,
    TrackId
FROM playlist_track
WHERE PlaylistId IS NOT NULL
AND NOT EXISTS (
    SELECT 1
    FROM playlists
    WHERE playlists.PlaylistId = playlist_track.PlaylistId
);

In [436]:
# Identify tracks not associated with a playlist
%%script sqlite3 --column --header chinook.db

-- Find tracks that are not associated with any playlist
SELECT
    TrackId
FROM tracks
WHERE NOT EXISTS (
    SELECT 1
    FROM playlist_track
    WHERE playlist_track.TrackId = tracks.TrackId
);

In [437]:
# Identify tracks not associated with a playlist
%%script sqlite3 --column --header chinook.db

-- Find playlists that are not associated with any track
SELECT
    PlaylistId
FROM playlists
WHERE NOT EXISTS (
    SELECT 1
    FROM playlist_track
    WHERE playlist_track.PlaylistId = playlists.PlaylistId
);

PlaylistId
----------
2         
4         
6         
7         


### 8. Employees

In [439]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from employees
limit 5

EmployeeId  LastName  FirstName  Title                ReportsTo  BirthDate            HireDate             Address              City      State  Country  PostalCode  Phone              Fax                Email                   
----------  --------  ---------  -------------------  ---------  -------------------  -------------------  -------------------  --------  -----  -------  ----------  -----------------  -----------------  ------------------------
1           Adams     Andrew     General Manager                 1962-02-18 00:00:00  2002-08-14 00:00:00  11120 Jasper Ave NW  Edmonton  AB     Canada   T5K 2N1     +1 (780) 428-9482  +1 (780) 428-3457  andrew@chinookcorp.com  
2           Edwards   Nancy      Sales Manager        1          1958-12-08 00:00:00  2002-05-01 00:00:00  825 8 Ave SW         Calgary   AB     Canada   T2P 2T3     +1 (403) 262-3443  +1 (403) 262-3322  nancy@chinookcorp.com   
3           Peacock   Jane       Sales Support Agent  2          1973-08-29 00:00:00

In [440]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    employeeId,
    FirstName,
    LastName
FROM employees
LIMIT 10;

EmployeeId  FirstName  LastName
----------  ---------  --------
1           Andrew     Adams   
2           Nancy      Edwards 
3           Jane       Peacock 
4           Margaret   Park    
5           Steve      Johnson 
6           Michael    Mitchell
7           Robert     King    
8           Laura      Callahan


In [441]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('employees') -- must have 'employees', not employees
PRAGMA table_info(employees);

cid  name        type          notnull  dflt_value  pk
---  ----------  ------------  -------  ----------  --
0    EmployeeId  INTEGER       1                    1 
1    LastName    NVARCHAR(20)  1                    0 
2    FirstName   NVARCHAR(20)  1                    0 
3    Title       NVARCHAR(30)  0                    0 
4    ReportsTo   INTEGER       0                    0 
5    BirthDate   DATETIME      0                    0 
6    HireDate    DATETIME      0                    0 
7    Address     NVARCHAR(70)  0                    0 
8    City        NVARCHAR(40)  0                    0 
9    State       NVARCHAR(40)  0                    0 
10   Country     NVARCHAR(40)  0                    0 
11   PostalCode  NVARCHAR(10)  0                    0 
12   Phone       NVARCHAR(24)  0                    0 
13   Fax         NVARCHAR(24)  0                    0 
14   Email       NVARCHAR(60)  0                    0 


In [442]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('employees')
WHERE pk > 0;

name        type     pk
----------  -------  --
EmployeeId  INTEGER  1 


In [443]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('employees');

id  seq  table      from       to          on_update  on_delete  match
--  ---  ---------  ---------  ----------  ---------  ---------  -----
0   0    employees  ReportsTo  EmployeeId  NO ACTION  NO ACTION  NONE 


In [444]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('employees')

total_columns
-------------
15           


In [445]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_employees
from employees

total_employees
---------------
8              


In [447]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM employees
WHERE FirstName IS NULL OR LastName IS NULL;

null_names
----------
0         


In [448]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN employeeId IS NULL THEN 1 ELSE 0 END) AS null_employee_ids,
    SUM(CASE WHEN LastName IS NULL THEN 1 ELSE 0 END) AS null_names
FROM employees;

total_rows  null_employee_ids  null_names
----------  -----------------  ----------
8           0                  0         


In [449]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN employeeId IS NULL OR LastName IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM employees;

total_rows  rows_with_nulls
----------  ---------------
8           0              


In [451]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN employeeId IS NULL THEN 1 ELSE 0 END) AS null_employee_ids,
    SUM(CASE WHEN LastName IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN employeeId IS NULL OR LastName IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM employees;

total_rows  null_employee_ids  null_names  rows_with_nulls
----------  -----------------  ----------  ---------------
8           0                  0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [453]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT employeeId, LastName
FROM employees
WHERE LastName LIKE 'A%';

EmployeeId  LastName
----------  --------
1           Adams   


In [454]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT employeeId, FirstName
FROM employees
WHERE FirstName LIKE 'A_%';

EmployeeId  FirstName
----------  ---------
1           Andrew   


In [456]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT employeeId, LastName
FROM employees
WHERE employeeId BETWEEN 1 AND 50
  AND LastName LIKE '%H%';

EmployeeId  LastName
----------  --------
5           Johnson 
6           Mitchell
8           Callahan


In [457]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT employeeId, LastName
FROM employees
WHERE employeeId = 1
   OR employeeId = 2
   OR employeeId = 3;

EmployeeId  LastName
----------  --------
1           Adams   
2           Edwards 
3           Peacock 


In [458]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM employees
LIMIT 5;

EmployeeId  LastName  FirstName  Title                ReportsTo  BirthDate            HireDate             Address              City      State  Country  PostalCode  Phone              Fax                Email                   
----------  --------  ---------  -------------------  ---------  -------------------  -------------------  -------------------  --------  -----  -------  ----------  -----------------  -----------------  ------------------------
1           Adams     Andrew     General Manager                 1962-02-18 00:00:00  2002-08-14 00:00:00  11120 Jasper Ave NW  Edmonton  AB     Canada   T5K 2N1     +1 (780) 428-9482  +1 (780) 428-3457  andrew@chinookcorp.com  
2           Edwards   Nancy      Sales Manager        1          1958-12-08 00:00:00  2002-05-01 00:00:00  825 8 Ave SW         Calgary   AB     Canada   T2P 2T3     +1 (403) 262-3443  +1 (403) 262-3322  nancy@chinookcorp.com   
3           Peacock   Jane       Sales Support Agent  2          1973-08-29 00:00:00

In [463]:
# Count unique employee names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT LastName) AS unique_employee_last_names
FROM employees;

unique_employee_last_names
--------------------------
8                         


In [464]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    employeeId,
    LastName
FROM employees
ORDER BY LastName ASC;

EmployeeId  LastName
----------  --------
1           Adams   
8           Callahan
2           Edwards 
5           Johnson 
7           King    
6           Mitchell
4           Park    
3           Peacock 


In [465]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    LastName,
    COUNT(*) AS occurrences
FROM employees
GROUP BY LastName
ORDER BY occurrences DESC;

LastName  occurrences
--------  -----------
Peacock   1          
Park      1          
Mitchell  1          
King      1          
Johnson   1          
Edwards   1          
Callahan  1          
Adams     1          


In [467]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT LastName, COUNT(*) AS occurrences
FROM employees
GROUP BY LastName
ORDER BY LastName
LIMIT 10;

LastName  occurrences
--------  -----------
Adams     1          
Callahan  1          
Edwards   1          
Johnson   1          
King      1          
Mitchell  1          
Park      1          
Peacock   1          


In [468]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_employees,
    SUM(employeeId) AS sum_employee_ids,
    AVG(employeeId) AS average_employee_id,
    MAX(employeeId) AS highest_employee_id,
    MIN(employeeId) AS lowest_employee_id
FROM employees;

total_employees  sum_employee_ids  average_employee_id  highest_employee_id  lowest_employee_id
---------------  ----------------  -------------------  -------------------  ------------------
8                36                4.5                  8                    1                 


In [469]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    employeeId,
    LastName,
    COUNT(*) AS occurrences
FROM employees
GROUP BY employeeId, LastName
HAVING COUNT(*) > 1;

In [471]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM employees
    GROUP BY employeeId, LastName, FirstName
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [472]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(employeeId) FROM employees

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(employeeId) FROM employees)
)

SELECT id AS missing_employee_id
FROM expected_ids
WHERE id NOT IN (
    SELECT employeeId FROM employees
);

#### Extras:

In [473]:
# Functions on Dates
%%script sqlite3 --column --header chinook.db
SELECT
    MIN(BirthDate) AS EarliestBirthDate,
    MAX(BirthDate) AS LatestBirthDate,
    MIN(HireDate) AS EarliestHireDate,
    MAX(HireDate) AS LatestHireDate
FROM employees;

EarliestBirthDate    LatestBirthDate      EarliestHireDate     LatestHireDate     
-------------------  -------------------  -------------------  -------------------
1947-09-19 00:00:00  1973-08-29 00:00:00  2002-04-01 00:00:00  2004-03-04 00:00:00


In [474]:
# EmployeeIDs of managers and number of reporting employees per manager
%%script sqlite3 --column --header chinook.db

SELECT
    ReportsTo,
    COUNT(*) AS DirectReports
FROM employees
WHERE ReportsTo IS NOT NULL
GROUP BY ReportsTo
ORDER BY DirectReports DESC;

ReportsTo  DirectReports
---------  -------------
2          3            
1          2            
6          2            


In [475]:
# Employees without a manager
%%script sqlite3 --column --header chinook.db
SELECT
    EmployeeId,
    FirstName,
    LastName,
    Title
FROM employees
WHERE ReportsTo IS NULL;

EmployeeId  FirstName  LastName  Title          
----------  ---------  --------  ---------------
1           Andrew     Adams     General Manager


In [476]:
# Self-reporting employees
%%script sqlite3 --column --header chinook.db
SELECT
    EmployeeId,
    FirstName,
    LastName,
    ReportsTo
FROM employees
WHERE EmployeeId = ReportsTo;

In [477]:
# Repeat employee names (not necessarily duplicate records)
%%script sqlite3 --column --header chinook.db
SELECT
    FirstName,
    LastName,
    COUNT(*) AS NameCount
FROM employees
GROUP BY FirstName, LastName
HAVING COUNT(*) > 1;

In [478]:
# Count total titles
%%script sqlite3 --column --header chinook.db
SELECT
    Title,
    COUNT(*) AS EmployeeCount
FROM employees
GROUP BY Title
ORDER BY EmployeeCount DESC;

Title                EmployeeCount
-------------------  -------------
Sales Support Agent  3            
IT Staff             2            
Sales Manager        1            
IT Manager           1            
General Manager      1            


In [479]:
# Distinct regions
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT
    Country,
    State,
    City
FROM employees
ORDER BY Country, State, City;

Country  State  City      
-------  -----  ----------
Canada   AB     Calgary   
Canada   AB     Edmonton  
Canada   AB     Lethbridge


In [506]:
# Check all columns for nulls
%%script sqlite3 --column --header chinook.db

-- Count NULL values in each employee column.
SELECT
    SUM(EmployeeId IS NULL) AS EmployeeId_NULLs,
    SUM(LastName IS NULL) AS LastName_NULLs,
    SUM(FirstName IS NULL) AS FirstName_NULLs,
    SUM(Title IS NULL) AS Title_NULLs,
    SUM(ReportsTo IS NULL) AS ReportsTo_NULLs,
    SUM(BirthDate IS NULL) AS BirthDate_NULLs,
    SUM(HireDate IS NULL) AS HireDate_NULLs,
    SUM(Address IS NULL) AS Address_NULLs,
    SUM(City IS NULL) AS City_NULLs,
    SUM(State IS NULL) AS State_NULLs,
    SUM(Country IS NULL) AS Country_NULLs,
    SUM(PostalCode IS NULL) AS PostalCode_NULLs,
    SUM(Phone IS NULL) AS Phone_NULLs,
    SUM(Fax IS NULL) AS Fax_NULLs,
    SUM(Email IS NULL) AS Email_NULLs
FROM employees;

EmployeeId_NULLs  LastName_NULLs  FirstName_NULLs  Title_NULLs  ReportsTo_NULLs  BirthDate_NULLs  HireDate_NULLs  Address_NULLs  City_NULLs  State_NULLs  Country_NULLs  PostalCode_NULLs  Phone_NULLs  Fax_NULLs  Email_NULLs
----------------  --------------  ---------------  -----------  ---------------  ---------------  --------------  -------------  ----------  -----------  -------------  ----------------  -----------  ---------  -----------
0                 0               0                0            1                0                0               0              0           0            0              0                 0            0          0          


### 9. Customers

In [480]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from customers
limit 5

CustomerId  FirstName  LastName     Company                                           Address                          City                 State  Country         PostalCode  Phone               Fax                 Email                     SupportRepId
----------  ---------  -----------  ------------------------------------------------  -------------------------------  -------------------  -----  --------------  ----------  ------------------  ------------------  ------------------------  ------------
1           Luís       Gonçalves    Embraer - Empresa Brasileira de Aeronáutica S.A.  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP     Brazil          12227-000   +55 (12) 3923-5555  +55 (12) 3923-5566  luisg@embraer.com.br      3           
2           Leonie     Köhler                                                         Theodor-Heuss-Straße 34          Stuttgart                   Germany         70174       +49 0711 2842222                        leonekohler@surfeu.de  

In [481]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    customerId,
    FirstName,
    LastName
FROM customers
LIMIT 10;

CustomerId  FirstName  LastName   
----------  ---------  -----------
1           Luís       Gonçalves  
2           Leonie     Köhler     
3           François   Tremblay   
4           Bjørn      Hansen     
5           František  Wichterlová
6           Helena     Holý       
7           Astrid     Gruber     
8           Daan       Peeters    
9           Kara       Nielsen    
10          Eduardo    Martins    


In [482]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('customers') -- must have 'customers', not customers
PRAGMA table_info(customers);

cid  name          type          notnull  dflt_value  pk
---  ------------  ------------  -------  ----------  --
0    CustomerId    INTEGER       1                    1 
1    FirstName     NVARCHAR(40)  1                    0 
2    LastName      NVARCHAR(20)  1                    0 
3    Company       NVARCHAR(80)  0                    0 
4    Address       NVARCHAR(70)  0                    0 
5    City          NVARCHAR(40)  0                    0 
6    State         NVARCHAR(40)  0                    0 
7    Country       NVARCHAR(40)  0                    0 
8    PostalCode    NVARCHAR(10)  0                    0 
9    Phone         NVARCHAR(24)  0                    0 
10   Fax           NVARCHAR(24)  0                    0 
11   Email         NVARCHAR(60)  1                    0 
12   SupportRepId  INTEGER       0                    0 


In [483]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('customers')
WHERE pk > 0;

name        type     pk
----------  -------  --
CustomerId  INTEGER  1 


In [484]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('customers');

id  seq  table      from          to          on_update  on_delete  match
--  ---  ---------  ------------  ----------  ---------  ---------  -----
0   0    employees  SupportRepId  EmployeeId  NO ACTION  NO ACTION  NONE 


In [485]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('customers')

total_columns
-------------
13           


In [486]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_customers
from customers

total_customers
---------------
59             


In [487]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_names
FROM customers
WHERE FirstName IS NULL OR LastName IS NULL;

null_names
----------
0         


In [488]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customerId IS NULL THEN 1 ELSE 0 END) AS null_customer_ids,
    SUM(CASE WHEN LastName IS NULL THEN 1 ELSE 0 END) AS null_names
FROM customers;

total_rows  null_customer_ids  null_names
----------  -----------------  ----------
59          0                  0         


In [489]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customerId IS NULL OR LastName IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM customers;

total_rows  rows_with_nulls
----------  ---------------
59          0              


In [490]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN customerId IS NULL THEN 1 ELSE 0 END) AS null_customer_ids,
    SUM(CASE WHEN LastName IS NULL THEN 1 ELSE 0 END) AS null_names,
    SUM(
        CASE
            WHEN customerId IS NULL OR LastName IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM customers;

total_rows  null_customer_ids  null_names  rows_with_nulls
----------  -----------------  ----------  ---------------
59          0                  0           0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [491]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT customerId, LastName
FROM customers
WHERE LastName LIKE 'A%';

CustomerId  LastName
----------  --------
12          Almeida 


In [492]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT customerId, FirstName
FROM customers
WHERE FirstName LIKE 'A_%';

CustomerId  FirstName
----------  ---------
7           Astrid   
11          Alexandre
32          Aaron    


In [493]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT customerId, LastName
FROM customers
WHERE customerId BETWEEN 1 AND 50
  AND LastName LIKE '%H%';

CustomerId  LastName   
----------  -----------
2           Köhler     
4           Hansen     
5           Wichterlová
6           Holý       
11          Rocha      
14          Philips    
16          Harris     
17          Smith      
21          Chase      
26          Cunningham 
32          Mitchell   
36          Schneider  
38          Schröder   
44          Hämäläinen 


In [494]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT customerId, LastName
FROM customers
WHERE customerId = 1
   OR customerId = 2
   OR customerId = 3;

CustomerId  LastName 
----------  ---------
1           Gonçalves
2           Köhler   
3           Tremblay 


In [495]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM customers
LIMIT 5;

CustomerId  FirstName  LastName     Company                                           Address                          City                 State  Country         PostalCode  Phone               Fax                 Email                     SupportRepId
----------  ---------  -----------  ------------------------------------------------  -------------------------------  -------------------  -----  --------------  ----------  ------------------  ------------------  ------------------------  ------------
1           Luís       Gonçalves    Embraer - Empresa Brasileira de Aeronáutica S.A.  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP     Brazil          12227-000   +55 (12) 3923-5555  +55 (12) 3923-5566  luisg@embraer.com.br      3           
2           Leonie     Köhler                                                         Theodor-Heuss-Straße 34          Stuttgart                   Germany         70174       +49 0711 2842222                        leonekohler@surfeu.de  

In [496]:
# Count unique customer names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT LastName) AS unique_customer_last_names
FROM customers;

unique_customer_last_names
--------------------------
59                        


In [497]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    customerId,
    LastName
FROM customers
ORDER BY LastName ASC;

CustomerId  LastName    
----------  ------------
12          Almeida     
28          Barnett     
39          Bernard     
18          Brooks      
29          Brown       
21          Chase       
26          Cunningham  
41          Dubois      
34          Fernandes   
30          Francis     
42          Girard      
1           Gonçalves   
23          Gordon      
19          Goyer       
27          Gray        
7           Gruber      
56          Gutiérrez   
4           Hansen      
16          Harris      
6           Holý        
53          Hughes      
44          Hämäläinen  
51          Johansson   
52          Jones       
45          Kovács      
2           Köhler      
22          Leacock     
40          Lefebvre    
47          Mancini     
10          Martins     
43          Mercier     
20          Miller      
32          Mitchell    
54          Murray      
50          Muñoz       
9           Nielsen     
46          O'Reilly    
58          Pareek      


In [498]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    LastName,
    COUNT(*) AS occurrences
FROM customers
GROUP BY LastName
ORDER BY occurrences DESC;

LastName      occurrences
------------  -----------
Zimmermann    1          
Wójcik        1          
Wichterlová   1          
Van der Berg  1          
Tremblay      1          
Taylor        1          
Sullivan      1          
Stevens       1          
Srivastava    1          
Smith         1          
Silk          1          
Schröder      1          
Schneider     1          
Sampaio       1          
Rojas         1          
Rocha         1          
Ramos         1          
Ralston       1          
Philips       1          
Peterson      1          
Peeters       1          
Pareek        1          
O'Reilly      1          
Nielsen       1          
Muñoz         1          
Murray        1          
Mitchell      1          
Miller        1          
Mercier       1          
Martins       1          
Mancini       1          
Lefebvre      1          
Leacock       1          
Köhler        1          
Kovács        1          
Jones         1          
Johansson   

In [499]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT LastName, COUNT(*) AS occurrences
FROM customers
GROUP BY LastName
ORDER BY LastName
LIMIT 10;

LastName    occurrences
----------  -----------
Almeida     1          
Barnett     1          
Bernard     1          
Brooks      1          
Brown       1          
Chase       1          
Cunningham  1          
Dubois      1          
Fernandes   1          
Francis     1          


In [500]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_customers,
    SUM(customerId) AS sum_customer_ids,
    AVG(customerId) AS average_customer_id,
    MAX(customerId) AS highest_customer_id,
    MIN(customerId) AS lowest_customer_id
FROM customers;

total_customers  sum_customer_ids  average_customer_id  highest_customer_id  lowest_customer_id
---------------  ----------------  -------------------  -------------------  ------------------
59               1770              30.0                 59                   1                 


In [501]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    customerId,
    LastName,
    COUNT(*) AS occurrences
FROM customers
GROUP BY customerId, LastName
HAVING COUNT(*) > 1;

In [502]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM customers
    GROUP BY customerId, LastName, FirstName
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [503]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(customerId) FROM customers

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(customerId) FROM customers)
)

SELECT id AS missing_customer_id
FROM expected_ids
WHERE id NOT IN (
    SELECT customerId FROM customers
);

#### Extras:

In [ ]:
# customerIDs of managers and number of reporting customers per manager
%%script sqlite3 --column --header chinook.db

SELECT
    ReportsTo,
    COUNT(*) AS DirectReports
FROM customers
WHERE ReportsTo IS NOT NULL
GROUP BY ReportsTo
ORDER BY DirectReports DESC;

ReportsTo  DirectReports
---------  -------------
2          3            
1          2            
6          2            


In [ ]:
# customers without a manager
%%script sqlite3 --column --header chinook.db
SELECT
    customerId,
    FirstName,
    LastName,
    Title
FROM customers
WHERE ReportsTo IS NULL;

EmployeeId  FirstName  LastName  Title          
----------  ---------  --------  ---------------
1           Andrew     Adams     General Manager


In [ ]:
# Self-reporting customers
%%script sqlite3 --column --header chinook.db
SELECT
    customerId,
    FirstName,
    LastName,
    ReportsTo
FROM customers
WHERE customerId = ReportsTo;

In [ ]:
# Repeat customer names (not necessarily duplicate records)
%%script sqlite3 --column --header chinook.db
SELECT
    FirstName,
    LastName,
    COUNT(*) AS NameCount
FROM customers
GROUP BY FirstName, LastName
HAVING COUNT(*) > 1;

In [ ]:
# Count total titles
%%script sqlite3 --column --header chinook.db
SELECT
    Title,
    COUNT(*) AS customerCount
FROM customers
GROUP BY Title
ORDER BY customerCount DESC;

Title                EmployeeCount
-------------------  -------------
Sales Support Agent  3            
IT Staff             2            
Sales Manager        1            
IT Manager           1            
General Manager      1            


In [ ]:
# Distinct regions
%%script sqlite3 --column --header chinook.db
SELECT DISTINCT
    Country,
    State,
    City
FROM customers
ORDER BY Country, State, City;

Country  State  City      
-------  -----  ----------
Canada   AB     Calgary   
Canada   AB     Edmonton  
Canada   AB     Lethbridge


In [505]:
# Check for all possible nulls
%%script sqlite3 --column --header chinook.db

SELECT
    SUM(CustomerId IS NULL) AS CustomerId_NULLs,
    SUM(FirstName IS NULL) AS FirstName_NULLs,
    SUM(LastName IS NULL) AS LastName_NULLs,
    SUM(Company IS NULL) AS Company_NULLs,
    SUM(Address IS NULL) AS Address_NULLs,
    SUM(City IS NULL) AS City_NULLs,
    SUM(State IS NULL) AS State_NULLs,
    SUM(Country IS NULL) AS Country_NULLs,
    SUM(PostalCode IS NULL) AS PostalCode_NULLs,
    SUM(Phone IS NULL) AS Phone_NULLs,
    SUM(Fax IS NULL) AS Fax_NULLs,
    SUM(Email IS NULL) AS Email_NULLs,
    SUM(SupportRepId IS NULL) AS SupportRepId_NULLs
FROM customers;

CustomerId_NULLs  FirstName_NULLs  LastName_NULLs  Company_NULLs  Address_NULLs  City_NULLs  State_NULLs  Country_NULLs  PostalCode_NULLs  Phone_NULLs  Fax_NULLs  Email_NULLs  SupportRepId_NULLs
----------------  ---------------  --------------  -------------  -------------  ----------  -----------  -------------  ----------------  -----------  ---------  -----------  ------------------
0                 0                0               49             0              0           29           0              4                 1            47         0            0                 


In [507]:
# Support representative frequency
%%script sqlite3 --column --header chinook.db
SELECT
    SupportRepId,
    COUNT(*) AS CustomerCount
FROM customers
GROUP BY SupportRepId
ORDER BY CustomerCount DESC;

SupportRepId  CustomerCount
------------  -------------
3             21           
4             20           
5             18           


In [509]:
# Distinct countries of origin
%%script sqlite3 --column --header chinook.db

SELECT DISTINCT
    Country
FROM customers
ORDER BY Country;

Country       
--------------
Argentina     
Australia     
Austria       
Belgium       
Brazil        
Canada        
Chile         
Czech Republic
Denmark       
Finland       
France        
Germany       
Hungary       
India         
Ireland       
Italy         
Netherlands   
Norway        
Poland        
Portugal      
Spain         
Sweden        
USA           
United Kingdom


In [508]:
# Customers by country
%%script sqlite3 --column --header chinook.db
SELECT
    Country,
    COUNT(*) AS CustomerCount
FROM customers
GROUP BY Country
ORDER BY CustomerCount DESC;

Country         CustomerCount
--------------  -------------
USA             13           
Canada          8            
France          5            
Brazil          5            
Germany         4            
United Kingdom  3            
Portugal        2            
India           2            
Czech Republic  2            
Sweden          1            
Spain           1            
Poland          1            
Norway          1            
Netherlands     1            
Italy           1            
Ireland         1            
Hungary         1            
Finland         1            
Denmark         1            
Chile           1            
Belgium         1            
Austria         1            
Australia       1            
Argentina       1            


### 10. Invoices

In [510]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from invoices
limit 5

InvoiceId  CustomerId  InvoiceDate          BillingAddress           BillingCity  BillingState  BillingCountry  BillingPostalCode  Total
---------  ----------  -------------------  -----------------------  -----------  ------------  --------------  -----------------  -----
1          2           2009-01-01 00:00:00  Theodor-Heuss-Straße 34  Stuttgart                  Germany         70174              1.98 
2          4           2009-01-02 00:00:00  Ullevålsveien 14         Oslo                       Norway          0171               3.96 
3          8           2009-01-03 00:00:00  Grétrystraat 63          Brussels                   Belgium         1000               5.94 
4          14          2009-01-06 00:00:00  8210 111 ST NW           Edmonton     AB            Canada          T6G 2C7            8.91 
5          23          2009-01-11 00:00:00  69 Salem Street          Boston       MA            USA             2113               13.86


In [512]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    invoiceId,
    Total
FROM invoices
LIMIT 10;

InvoiceId  Total
---------  -----
1          1.98 
2          3.96 
3          5.94 
4          8.91 
5          13.86
6          0.99 
7          1.98 
8          1.98 
9          3.96 
10         5.94 


In [513]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('invoices') -- must have 'invoices', not invoices
PRAGMA table_info(invoices);

cid  name               type           notnull  dflt_value  pk
---  -----------------  -------------  -------  ----------  --
0    InvoiceId          INTEGER        1                    1 
1    CustomerId         INTEGER        1                    0 
2    InvoiceDate        DATETIME       1                    0 
3    BillingAddress     NVARCHAR(70)   0                    0 
4    BillingCity        NVARCHAR(40)   0                    0 
5    BillingState       NVARCHAR(40)   0                    0 
6    BillingCountry     NVARCHAR(40)   0                    0 
7    BillingPostalCode  NVARCHAR(10)   0                    0 
8    Total              NUMERIC(10,2)  1                    0 


In [514]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('invoices')
WHERE pk > 0;

name       type     pk
---------  -------  --
InvoiceId  INTEGER  1 


In [515]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('invoices');

id  seq  table      from        to          on_update  on_delete  match
--  ---  ---------  ----------  ----------  ---------  ---------  -----
0   0    customers  CustomerId  CustomerId  NO ACTION  NO ACTION  NONE 


In [516]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('invoices')

total_columns
-------------
9            


In [517]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_invoices
from invoices

total_invoices
--------------
412           


In [519]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_totals
FROM invoices
WHERE Total IS NULL;

null_totals
-----------
0          


In [520]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoiceId IS NULL THEN 1 ELSE 0 END) AS null_invoice_ids,
    SUM(CASE WHEN total IS NULL THEN 1 ELSE 0 END) AS null_totals
FROM invoices;

total_rows  null_invoice_ids  null_totals
----------  ----------------  -----------
412         0                 0          


In [521]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoiceId IS NULL OR total IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM invoices;

total_rows  rows_with_nulls
----------  ---------------
412         0              


In [522]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoiceId IS NULL THEN 1 ELSE 0 END) AS null_invoice_ids,
    SUM(CASE WHEN total IS NULL THEN 1 ELSE 0 END) AS null_totals,
    SUM(
        CASE
            WHEN invoiceId IS NULL OR total IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM invoices;

total_rows  null_invoice_ids  null_totals  rows_with_nulls
----------  ----------------  -----------  ---------------
412         0                 0            0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [524]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT invoiceId, BillingAddress
FROM invoices
WHERE BillingAddress LIKE 'A%';

InvoiceId  BillingAddress                 
---------  -------------------------------
57         Av. Paulista, 2022             
68         Av. Paulista, 2022             
98         Av. Brigadeiro Faria Lima, 2170
121        Av. Brigadeiro Faria Lima, 2170
123        Av. Paulista, 2022             
143        Av. Brigadeiro Faria Lima, 2170
195        Av. Brigadeiro Faria Lima, 2170
252        Av. Paulista, 2022             
275        Av. Paulista, 2022             
297        Av. Paulista, 2022             
316        Av. Brigadeiro Faria Lima, 2170
327        Av. Brigadeiro Faria Lima, 2170
349        Av. Paulista, 2022             
382        Av. Brigadeiro Faria Lima, 2170


In [525]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT invoiceId, BillingAddress
FROM invoices
WHERE BillingAddress LIKE 'A_%';

InvoiceId  BillingAddress                 
---------  -------------------------------
57         Av. Paulista, 2022             
68         Av. Paulista, 2022             
98         Av. Brigadeiro Faria Lima, 2170
121        Av. Brigadeiro Faria Lima, 2170
123        Av. Paulista, 2022             
143        Av. Brigadeiro Faria Lima, 2170
195        Av. Brigadeiro Faria Lima, 2170
252        Av. Paulista, 2022             
275        Av. Paulista, 2022             
297        Av. Paulista, 2022             
316        Av. Brigadeiro Faria Lima, 2170
327        Av. Brigadeiro Faria Lima, 2170
349        Av. Paulista, 2022             
382        Av. Brigadeiro Faria Lima, 2170


In [526]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT invoiceId, BillingAddress
FROM invoices
WHERE invoiceId BETWEEN 1 AND 50
  AND BillingAddress LIKE '%H%';

InvoiceId  BillingAddress           
---------  -------------------------
1          Theodor-Heuss-Straße 34  
8          8, Rue Hanovre           
9          9, Place Louis Barthou   
10         3 Chatham Street         
11         202 Hoxton Street        
12         Theodor-Heuss-Straße 34  
13         1600 Amphitheatre Parkway
16         801 W 4th Street         
18         194A Chain Lake Drive    
19         8, Rue Hanovre           
23         3,Raj Bhavan Road        
25         Rua Dr. Falcão Filho, 155
31         9, Place Louis Barthou   
32         Lijnbaansgracht 120bg    
38         801 W 4th Street         
45         3,Raj Bhavan Road        


In [527]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT invoiceId, BillingAddress
FROM invoices
WHERE invoiceId = 1
   OR invoiceId = 2
   OR invoiceId = 3;

InvoiceId  BillingAddress         
---------  -----------------------
1          Theodor-Heuss-Straße 34
2          Ullevålsveien 14       
3          Grétrystraat 63        


In [528]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM invoices
LIMIT 5;

InvoiceId  CustomerId  InvoiceDate          BillingAddress           BillingCity  BillingState  BillingCountry  BillingPostalCode  Total
---------  ----------  -------------------  -----------------------  -----------  ------------  --------------  -----------------  -----
1          2           2009-01-01 00:00:00  Theodor-Heuss-Straße 34  Stuttgart                  Germany         70174              1.98 
2          4           2009-01-02 00:00:00  Ullevålsveien 14         Oslo                       Norway          0171               3.96 
3          8           2009-01-03 00:00:00  Grétrystraat 63          Brussels                   Belgium         1000               5.94 
4          14          2009-01-06 00:00:00  8210 111 ST NW           Edmonton     AB            Canada          T6G 2C7            8.91 
5          23          2009-01-11 00:00:00  69 Salem Street          Boston       MA            USA             2113               13.86


In [529]:
# Count unique invoice names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT BillingAddress) AS unique_invoice_BillingAddresses
FROM invoices;

unique_invoice_BillingAddresses
-------------------------------
59                             


In [531]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    invoiceId,
    BillingAddress
FROM invoices
ORDER BY BillingAddress ASC;

InvoiceId  BillingAddress                          
---------  ----------------------------------------
15         1 Infinite Loop                         
26         1 Infinite Loop                         
81         1 Infinite Loop                         
210        1 Infinite Loop                         
233        1 Infinite Loop                         
255        1 Infinite Loop                         
307        1 Infinite Loop                         
14         1 Microsoft Way                         
37         1 Microsoft Way                         
59         1 Microsoft Way                         
111        1 Microsoft Way                         
232        1 Microsoft Way                         
243        1 Microsoft Way                         
298        1 Microsoft Way                         
39         1033 N Park Ave                         
168        1033 N Park Ave                         
191        1033 N Park Ave                         
213        1

In [532]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    BillingAddress,
    COUNT(*) AS occurrences
FROM invoices
GROUP BY BillingAddress
ORDER BY occurrences DESC;

BillingAddress                            occurrences
----------------------------------------  -----------
Via Degli Scipioni, 43                    7          
Ullevålsveien 14                          7          
Theodor-Heuss-Straße 34                   7          
Tauentzienstraße 8                        7          
Sønder Boulevard 51                       7          
Rua dos Campeões Europeus de Viena, 4350  7          
Rua da Assunção 53                        7          
Rua Dr. Falcão Filho, 155                 7          
Rotenturmstraße 4, 1010 Innere Stadt      7          
Rilská 3174/6                             7          
Qe 7 Bloco G                              7          
Praça Pio X, 119                          7          
Porthaninkatu 9                           7          
Ordynacka 10                              7          
Lijnbaansgracht 120bg                     7          
Klanova 9/506                             7          
Grétrystraat 63             

In [533]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT BillingAddress, COUNT(*) AS occurrences
FROM invoices
GROUP BY BillingAddress
ORDER BY BillingAddress
LIMIT 10;

BillingAddress             occurrences
-------------------------  -----------
1 Infinite Loop            7          
1 Microsoft Way            7          
1033 N Park Ave            7          
11, Place Bellecour        7          
110 Raeburn Pl             7          
113 Lupus St               7          
12,Community Centre        7          
120 S Orange Ave           7          
1498 rue Bélanger          7          
1600 Amphitheatre Parkway  7          


In [534]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_invoices,
    SUM(invoiceId) AS sum_invoice_ids,
    AVG(invoiceId) AS average_invoice_id,
    MAX(invoiceId) AS highest_invoice_id,
    MIN(invoiceId) AS lowest_invoice_id
FROM invoices;

total_invoices  sum_invoice_ids  average_invoice_id  highest_invoice_id  lowest_invoice_id
--------------  ---------------  ------------------  ------------------  -----------------
412             85078            206.5               412                 1                


In [536]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    invoiceId,
    BillingAddress,
    COUNT(*) AS occurrences
FROM invoices
GROUP BY invoiceId, BillingAddress
HAVING COUNT(*) > 1;

In [537]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM invoices
    GROUP BY invoiceId, BillingAddress
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [538]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(invoiceId) FROM invoices

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(invoiceId) FROM invoices)
)

SELECT id AS missing_invoice_id
FROM expected_ids
WHERE id NOT IN (
    SELECT invoiceId FROM invoices
);

### 11. Invoice_Items

In [539]:
# Select the first 10 entries from table
%%script sqlite3 --column --header chinook.db
select *
from invoice_items
limit 5

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity
-------------  ---------  -------  ---------  --------
1              1          2        0.99       1       
2              1          4        0.99       1       
3              2          6        0.99       1       
4              2          8        0.99       1       
5              2          10       0.99       1       


In [540]:
# Select specific columns
%%script sqlite3 --column --header chinook.db

SELECT
    invoicelineId,
    UnitPrice
FROM invoice_items
LIMIT 10;

InvoiceLineId  UnitPrice
-------------  ---------
1              0.99     
2              0.99     
3              0.99     
4              0.99     
5              0.99     
6              0.99     
7              0.99     
8              0.99     
9              0.99     
10             0.99     


In [541]:
# Check table metainfo to view columns and info
%%script sqlite3 --column --header chinook.db
-- select * from pragma_table_info('invoice_items') -- must have 'invoice_items', not invoice_items
PRAGMA table_info(invoice_items);

cid  name           type           notnull  dflt_value  pk
---  -------------  -------------  -------  ----------  --
0    InvoiceLineId  INTEGER        1                    1 
1    InvoiceId      INTEGER        1                    0 
2    TrackId        INTEGER        1                    0 
3    UnitPrice      NUMERIC(10,2)  1                    0 
4    Quantity       INTEGER        1                    0 


In [542]:
# Check table metainfo for id columns
%%script sqlite3 --column --header chinook.db
SELECT name, type, pk
FROM pragma_table_info('invoice_items')
WHERE pk > 0;

name           type     pk
-------------  -------  --
InvoiceLineId  INTEGER  1 


In [543]:
# Check foreign key list (metainfo) for foreign key columns
%%script sqlite3 --column --header chinook.db
PRAGMA foreign_key_list('invoice_items');

id  seq  table     from       to         on_update  on_delete  match
--  ---  --------  ---------  ---------  ---------  ---------  -----
0   0    tracks    TrackId    TrackId    NO ACTION  NO ACTION  NONE 
1   0    invoices  InvoiceId  InvoiceId  NO ACTION  NO ACTION  NONE 


In [544]:
# Get number of columns in table
%%script sqlite3 --column --header chinook.db
select count(*) as total_columns
from pragma_table_info('invoice_items')

total_columns
-------------
5            


In [545]:
# Count the number of rows
%%script sqlite3 --column --header chinook.db
select count(*) as total_invoice_items
from invoice_items

total_invoice_items
-------------------
2240               


In [548]:
# Count rows where name is null
%%script sqlite3 --column --header chinook.db
SELECT COUNT(*) AS null_unit_prices
FROM invoice_items
WHERE UnitPrice IS NULL;

null_unit_prices
----------------
0               


In [550]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoicelineId IS NULL THEN 1 ELSE 0 END) AS null_invoice_line_ids,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS null_unit_prices
FROM invoice_items;

total_rows  null_invoice_line_ids  null_unit_prices
----------  ---------------------  ----------------
2240        0                      0               


In [551]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoicelineId IS NULL OR unitprice IS NULL THEN 1 ELSE 0 END) AS rows_with_nulls
FROM invoice_items;

total_rows  rows_with_nulls
----------  ---------------
2240        0              


In [554]:
# Count nulls in each column
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN invoicelineId IS NULL THEN 1 ELSE 0 END) AS null_invoice_line_ids,
    SUM(CASE WHEN unitprice IS NULL THEN 1 ELSE 0 END) AS null_unit_prices,
    SUM(
        CASE
            WHEN invoicelineId IS NULL OR unitprice IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_with_nulls
FROM invoice_items;

total_rows  null_invoice_line_ids  null_unit_prices  rows_with_nulls
----------  ---------------------  ----------------  ---------------
2240        0                      0                 0              


#### Filtering and Searching Records: WHERE, AND, OR, LIKE, BETWEEN, LIMIT

In [557]:
# WHERE and LIKE (%)
%%script sqlite3 --column --header chinook.db
SELECT invoicelineId, UnitPrice
FROM invoice_items
WHERE UnitPrice LIKE '0%';

InvoiceLineId  UnitPrice
-------------  ---------
1              0.99     
2              0.99     
3              0.99     
4              0.99     
5              0.99     
6              0.99     
7              0.99     
8              0.99     
9              0.99     
10             0.99     
11             0.99     
12             0.99     
13             0.99     
14             0.99     
15             0.99     
16             0.99     
17             0.99     
18             0.99     
19             0.99     
20             0.99     
21             0.99     
22             0.99     
23             0.99     
24             0.99     
25             0.99     
26             0.99     
27             0.99     
28             0.99     
29             0.99     
30             0.99     
31             0.99     
32             0.99     
33             0.99     
34             0.99     
35             0.99     
36             0.99     
37             0.99     
38             0.99     


In [558]:
# WHERE and LIKE (_%)
%%script sqlite3 --column --header chinook.db
SELECT invoicelineId, UnitPrice
FROM invoice_items
WHERE UnitPrice LIKE '1_%';

InvoiceLineId  UnitPrice
-------------  ---------
468            1.99     
469            1.99     
470            1.99     
471            1.99     
472            1.99     
473            1.99     
474            1.99     
475            1.99     
476            1.99     
477            1.99     
478            1.99     
479            1.99     
480            1.99     
481            1.99     
482            1.99     
522            1.99     
523            1.99     
524            1.99     
525            1.99     
526            1.99     
527            1.99     
528            1.99     
529            1.99     
530            1.99     
531            1.99     
532            1.99     
533            1.99     
534            1.99     
553            1.99     
554            1.99     
563            1.99     
1042           1.99     
1043           1.99     
1044           1.99     
1045           1.99     
1046           1.99     
1047           1.99     
1048           1.99     


In [561]:
# AND and BETWEEN
%%script sqlite3 --column --header chinook.db
SELECT invoicelineId, UnitPrice
FROM invoice_items
WHERE invoicelineId BETWEEN 1 AND 50
  AND UnitPrice LIKE '%9%';

InvoiceLineId  UnitPrice
-------------  ---------
1              0.99     
2              0.99     
3              0.99     
4              0.99     
5              0.99     
6              0.99     
7              0.99     
8              0.99     
9              0.99     
10             0.99     
11             0.99     
12             0.99     
13             0.99     
14             0.99     
15             0.99     
16             0.99     
17             0.99     
18             0.99     
19             0.99     
20             0.99     
21             0.99     
22             0.99     
23             0.99     
24             0.99     
25             0.99     
26             0.99     
27             0.99     
28             0.99     
29             0.99     
30             0.99     
31             0.99     
32             0.99     
33             0.99     
34             0.99     
35             0.99     
36             0.99     
37             0.99     
38             0.99     


In [563]:
# OR
%%script sqlite3 --column --header chinook.db
SELECT invoicelineId, UnitPrice
FROM invoice_items
WHERE invoicelineId = 1
   OR invoicelineId = 2
   OR invoicelineId = 3;

InvoiceLineId  UnitPrice
-------------  ---------
1              0.99     
2              0.99     
3              0.99     


In [564]:
# LIMIT
%%script sqlite3 --column --header chinook.db
SELECT *
FROM invoice_items
LIMIT 5;

InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity
-------------  ---------  -------  ---------  --------
1              1          2        0.99       1       
2              1          4        0.99       1       
3              2          6        0.99       1       
4              2          8        0.99       1       
5              2          10       0.99       1       


In [565]:
# Count unique invoice names
%%script sqlite3 --column --header chinook.db
SELECT
    COUNT(DISTINCT UnitPrice) AS unique_invoice_UnitPrices
FROM invoice_items;

unique_invoice_UnitPrices
-------------------------
2                        


In [566]:
# ORDER BY
%%script sqlite3 --column --header chinook.db
SELECT
    invoicelineId,
    UnitPrice
FROM invoice_items
ORDER BY UnitPrice ASC;

InvoiceLineId  UnitPrice
-------------  ---------
1              0.99     
2              0.99     
3              0.99     
4              0.99     
5              0.99     
6              0.99     
7              0.99     
8              0.99     
9              0.99     
10             0.99     
11             0.99     
12             0.99     
13             0.99     
14             0.99     
15             0.99     
16             0.99     
17             0.99     
18             0.99     
19             0.99     
20             0.99     
21             0.99     
22             0.99     
23             0.99     
24             0.99     
25             0.99     
26             0.99     
27             0.99     
28             0.99     
29             0.99     
30             0.99     
31             0.99     
32             0.99     
33             0.99     
34             0.99     
35             0.99     
36             0.99     
37             0.99     
38             0.99     


In [567]:
# GROUP BY
%%script sqlite3 --column --header chinook.db

SELECT
    UnitPrice,
    COUNT(*) AS occurrences
FROM invoice_items
GROUP BY UnitPrice
ORDER BY occurrences DESC;

UnitPrice  occurrences
---------  -----------
0.99       2129       
1.99       111        


In [568]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT UnitPrice, COUNT(*) AS occurrences
FROM invoice_items
GROUP BY UnitPrice
ORDER BY UnitPrice
LIMIT 10;

UnitPrice  occurrences
---------  -----------
0.99       2129       
1.99       111        


In [569]:
# Aggregate functions
%%script sqlite3 --column --header chinook.db

SELECT
    COUNT(*) AS total_invoice_items,
    SUM(invoicelineId) AS sum_invoice_ids,
    AVG(invoicelineId) AS average_invoice_id,
    MAX(invoicelineId) AS highest_invoice_id,
    MIN(invoicelineId) AS lowest_invoice_id
FROM invoice_items;

total_invoice_items  sum_invoice_ids  average_invoice_id  highest_invoice_id  lowest_invoice_id
-------------------  ---------------  ------------------  ------------------  -----------------
2240                 2509920          1120.5              2240                1                


In [571]:
# Count categories
%%script sqlite3 --column --header chinook.db
SELECT
    invoicelineId,
    UnitPrice,
    COUNT(*) AS occurrences
FROM invoice_items
GROUP BY invoicelineId, UnitPrice
HAVING COUNT(*) > 1;

In [ ]:
# Count excess duplicate rows
%%script sqlite3 --column --header chinook.db

SELECT
    COALESCE(SUM(occurrences - 1), 0) AS duplicate_rows
FROM (
    SELECT COUNT(*) AS occurrences
    FROM invoice_items
    GROUP BY invoicelineId, BillingAddress
    HAVING COUNT(*) > 1
);

duplicate_rows
--------------
0             


In [572]:
# Look for missing rows
%%script sqlite3 --column --header chinook.db
WITH RECURSIVE expected_ids(id) AS (
    SELECT MIN(invoicelineId) FROM invoice_items

    UNION ALL

    SELECT id + 1
    FROM expected_ids
    WHERE id < (SELECT MAX(invoicelineId) FROM invoice_items)
)

SELECT id AS missing_invoice_id
FROM expected_ids
WHERE id NOT IN (
    SELECT invoicelineId FROM invoice_items
);

## Phase 2. Map the Terrain (Get Familiar with the Data)



Data never exists in a vacuum. By writing queries, you will uncover the shape, structure, and quirks of the Chinook dataset. Again, using the SQL statements below, you'll learn how the tables relate to one another, what a single row represents in different contexts, and where specific types of information live.  Here's where you should get familiar with the ERD and join tables in various ways.



The following are some SQL commands that demonstrate some of the relationships between tables in the Chinook Dataset.

#### Employees with Employees

In [14]:
# Select some fields from the employees table
%%script sqlite3 --column --header chinook.db
select EmployeeId, LastName, FirstName, Title, ReportsTo
from employees

EmployeeId  LastName  FirstName  Title                ReportsTo
----------  --------  ---------  -------------------  ---------
1           Adams     Andrew     General Manager               
2           Edwards   Nancy      Sales Manager        1        
3           Peacock   Jane       Sales Support Agent  2        
4           Park      Margaret   Sales Support Agent  2        
5           Johnson   Steve      Sales Support Agent  2        
6           Mitchell  Michael    IT Manager           1        
7           King      Robert     IT Staff             6        
8           Callahan  Laura      IT Staff             6        


In [31]:
# Employees with Itself

# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select
  employee.EmployeeId as EmpId,
  employee.LastName as Last,
  employee.FirstName as First,
  employee.Title,
  employee.ReportsTo as RepsTo,
  manager.EmployeeId as SupId,
  manager.LastName as Last,
  manager.FirstName as First
from employees as employee
inner join employees as manager
on employee.ReportsTo = manager.EmployeeID

EmpId  Last      First     Title                RepsTo  SupId  Last      First  
-----  --------  --------  -------------------  ------  -----  --------  -------
2      Edwards   Nancy     Sales Manager        1       1      Adams     Andrew 
3      Peacock   Jane      Sales Support Agent  2       2      Edwards   Nancy  
4      Park      Margaret  Sales Support Agent  2       2      Edwards   Nancy  
5      Johnson   Steve     Sales Support Agent  2       2      Edwards   Nancy  
6      Mitchell  Michael   IT Manager           1       1      Adams     Andrew 
7      King      Robert    IT Staff             6       6      Mitchell  Michael
8      Callahan  Laura     IT Staff             6       6      Mitchell  Michael


In [20]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select employee.EmployeeId, employee.LastName, employee.ReportsTo, manager.EmployeeId as ManagerId, manager.LastName
from employees as employee
inner join employees as manager
on employee.ReportsTo = manager.EmployeeID

EmployeeId  LastName  ReportsTo  ManagerId  LastName
----------  --------  ---------  ---------  --------
2           Edwards   1          1          Adams   
3           Peacock   2          2          Edwards 
4           Park      2          2          Edwards 
5           Johnson   2          2          Edwards 
6           Mitchell  1          1          Adams   
7           King      6          6          Mitchell
8           Callahan  6          6          Mitchell


In [22]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select employee.EmployeeId, employee.LastName, employee.ReportsTo, manager.EmployeeId as ManagerId, manager.LastName
from employees as employee
left join employees as manager
on employee.ReportsTo = manager.EmployeeID ;

EmployeeId  LastName  ReportsTo  ManagerId  LastName
----------  --------  ---------  ---------  --------
1           Adams                                   
2           Edwards   1          1          Adams   
3           Peacock   2          2          Edwards 
4           Park      2          2          Edwards 
5           Johnson   2          2          Edwards 
6           Mitchell  1          1          Adams   
7           King      6          6          Mitchell
8           Callahan  6          6          Mitchell


In [23]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select employee.EmployeeId, employee.LastName, employee.ReportsTo, manager.EmployeeId as ManagerId, manager.LastName
from employees as employee
right join employees as manager
on employee.ReportsTo = manager.EmployeeID ;

EmployeeId  LastName  ReportsTo  ManagerId  LastName
----------  --------  ---------  ---------  --------
2           Edwards   1          1          Adams   
3           Peacock   2          2          Edwards 
4           Park      2          2          Edwards 
5           Johnson   2          2          Edwards 
6           Mitchell  1          1          Adams   
7           King      6          6          Mitchell
8           Callahan  6          6          Mitchell
                                 3          Peacock 
                                 4          Park    
                                 5          Johnson 
                                 7          King    
                                 8          Callahan


In [24]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select employee.EmployeeId, employee.LastName, employee.ReportsTo, manager.EmployeeId as ManagerId, manager.LastName
from employees as employee
full outer join employees as manager
on employee.ReportsTo = manager.EmployeeID ;

EmployeeId  LastName  ReportsTo  ManagerId  LastName
----------  --------  ---------  ---------  --------
1           Adams                                   
2           Edwards   1          1          Adams   
3           Peacock   2          2          Edwards 
4           Park      2          2          Edwards 
5           Johnson   2          2          Edwards 
6           Mitchell  1          1          Adams   
7           King      6          6          Mitchell
8           Callahan  6          6          Mitchell
                                 3          Peacock 
                                 4          Park    
                                 5          Johnson 
                                 7          King    
                                 8          Callahan


In [25]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select employee.EmployeeId, employee.LastName, employee.ReportsTo, manager.EmployeeId as ManagerId, manager.LastName
from employees as employee
join employees as manager
limit 10 ;

EmployeeId  LastName  ReportsTo  ManagerId  LastName
----------  --------  ---------  ---------  --------
1           Adams                1          Adams   
1           Adams                2          Edwards 
1           Adams                3          Peacock 
1           Adams                4          Park    
1           Adams                5          Johnson 
1           Adams                6          Mitchell
1           Adams                7          King    
1           Adams                8          Callahan
2           Edwards   1          1          Adams   
2           Edwards   1          2          Edwards 


#### Employee with Customer

In [25]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select *
from customers
limit 5;
select count(*) as total_cust
from customers;

CustomerId  FirstName  LastName     Company                                           Address                          City                 State  Country         PostalCode  Phone               Fax                 Email                     SupportRepId
----------  ---------  -----------  ------------------------------------------------  -------------------------------  -------------------  -----  --------------  ----------  ------------------  ------------------  ------------------------  ------------
1           Luís       Gonçalves    Embraer - Empresa Brasileira de Aeronáutica S.A.  Av. Brigadeiro Faria Lima, 2170  São José dos Campos  SP     Brazil          12227-000   +55 (12) 3923-5555  +55 (12) 3923-5566  luisg@embraer.com.br      3           
2           Leonie     Köhler                                                         Theodor-Heuss-Straße 34          Stuttgart                   Germany         70174       +49 0711 2842222                        leonekohler@surfeu.de  

In [30]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select
  emp.EmployeeId as EmpId,
  emp.LastName as Last,
  emp.FirstName as First,
  cust.supportrepid,
  cust.LastName as Last,
  cust.FirstName as First
from employees as emp
inner join customers as cust
on emp.employeeid = cust.supportrepid
order by empid ;

EmpId  Last     First     SupportRepId  Last          First    
-----  -------  --------  ------------  ------------  ---------
3      Peacock  Jane      3             Gonçalves     Luís     
3      Peacock  Jane      3             Tremblay      François 
3      Peacock  Jane      3             Almeida       Roberto  
3      Peacock  Jane      3             Peterson      Jennifer 
3      Peacock  Jane      3             Brooks        Michelle 
3      Peacock  Jane      3             Goyer         Tim      
3      Peacock  Jane      3             Ralston       Frank    
3      Peacock  Jane      3             Brown         Robert   
3      Peacock  Jane      3             Francis       Edward   
3      Peacock  Jane      3             Sullivan      Ellie    
3      Peacock  Jane      3             Zimmermann    Fynn     
3      Peacock  Jane      3             Schröder      Niklas   
3      Peacock  Jane      3             Girard        Wyatt    
3      Peacock  Jane      3             

In [27]:
# Relate employees to customers,
%%script sqlite3 --column --header chinook.db
select
  emp.EmployeeId as EmpId,
  emp.LastName as Last,
  emp.FirstName as First,
  cust.supportrepid as RepId,
  cust.LastName as Last,
  cust.FirstName as First
from employees as emp
left join customers as cust
on emp.employeeid = cust.supportrepid;

EmpId  Last      First     RepId  Last          First    
-----  --------  --------  -----  ------------  ---------
1      Adams     Andrew                                  
2      Edwards   Nancy                                   
3      Peacock   Jane      3      Gonçalves     Luís     
3      Peacock   Jane      3      Tremblay      François 
3      Peacock   Jane      3      Almeida       Roberto  
3      Peacock   Jane      3      Peterson      Jennifer 
3      Peacock   Jane      3      Brooks        Michelle 
3      Peacock   Jane      3      Goyer         Tim      
3      Peacock   Jane      3      Ralston       Frank    
3      Peacock   Jane      3      Brown         Robert   
3      Peacock   Jane      3      Francis       Edward   
3      Peacock   Jane      3      Sullivan      Ellie    
3      Peacock   Jane      3      Zimmermann    Fynn     
3      Peacock   Jane      3      Schröder      Niklas   
3      Peacock   Jane      3      Girard        Wyatt    
3      Peacock

In [17]:
# Select employees and their managers
%%script sqlite3 --column --header chinook.db
select
  emp.EmployeeId as EmpId,
  emp.LastName as Last,
  emp.ReportsTo as RepsTo,
  sup.EmployeeId as SupId,
  sup.LastName as Last
from employees as emp
left join employees as sup
on emp.ReportsTo = sup.EmployeeID ;

EmpId  Last      RepsTo  SupId  Last    
-----  --------  ------  -----  --------
1      Adams                            
2      Edwards   1       1      Adams   
3      Peacock   2       2      Edwards 
4      Park      2       2      Edwards 
5      Johnson   2       2      Edwards 
6      Mitchell  1       1      Adams   
7      King      6       6      Mitchell
8      Callahan  6       6      Mitchell


In [31]:
# Employee with Customer
%%script sqlite3 --column --header chinook.db
with employee as (
select
  employeeid as EmpId,
  lastname as Last,
  firstname as First,
  reportsto as RepsTo
from employees
)
select *
from employee ;


EmpId  Last      First     RepsTo
-----  --------  --------  ------
1      Adams     Andrew          
2      Edwards   Nancy     1     
3      Peacock   Jane      2     
4      Park      Margaret  2     
5      Johnson   Steve     2     
6      Mitchell  Michael   1     
7      King      Robert    6     
8      Callahan  Laura     6     


In [36]:
# Relate employees to customers,
%%script sqlite3 --column --header chinook.db
with
cust_emp as (
select
  cust.LastName as custLast,
  cust.FirstName as custFirst,
  cust.supportrepid as repId,
  emp.EmployeeId as empId,
  emp.LastName as empLast,
  emp.FirstName as empFirst
from employees as emp
left join customers as cust
on emp.employeeid = cust.supportrepid
),
emp_sup as (
select
  emp.EmployeeId as empId,
  emp.LastName as empLast,
  emp.FirstName as emoFirst,
  emp.ReportsTo as repsTo,
  sup.EmployeeId as supId,
  sup.LastName as supLast,
  sup.FirstName as supFirst
from employees as emp
left join employees as sup
on emp.ReportsTo = sup.EmployeeID
),
cust_emp_sup as (
select
  cust_emp.custLast as Last,
  cust_emp.custFirst as First,
  cust_emp.repid as RepId,
  cust_emp.empId as EmpId,
  cust_emp.empLast as Last,
  cust_emp.empFirst as First,
  emp_sup.repsTo as RepsTo,
  emp_sup.supId as SupId,
  emp_sup.supLast as Last,
  emp_sup.supFirst as First
from cust_emp
inner join emp_sup
on cust_emp.empId = emp_sup.empId
)
select *
from cust_emp_sup ;

Last          First      RepId  EmpId  Last:1    First:1   RepsTo  SupId  Last:2    First:2
------------  ---------  -----  -----  --------  --------  ------  -----  --------  -------
                                1      Adams     Andrew                                    
                                2      Edwards   Nancy     1       1      Adams     Andrew 
                                6      Mitchell  Michael   1       1      Adams     Andrew 
Gonçalves     Luís       3      3      Peacock   Jane      2       2      Edwards   Nancy  
Tremblay      François   3      3      Peacock   Jane      2       2      Edwards   Nancy  
Almeida       Roberto    3      3      Peacock   Jane      2       2      Edwards   Nancy  
Peterson      Jennifer   3      3      Peacock   Jane      2       2      Edwards   Nancy  
Brooks        Michelle   3      3      Peacock   Jane      2       2      Edwards   Nancy  
Goyer         Tim        3      3      Peacock   Jane      2       2      Edward

#### Albums to Artists

In [64]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
select al.albumid as Id, al.title as Album, ar.name as Artist
from albums as al
inner join artists as ar
on al.artistid = ar.artistid
limit 5;

Id  Album                                  Artist   
--  -------------------------------------  ---------
1   For Those About To Rock We Salute You  AC/DC    
2   Balls to the Wall                      Accept   
3   Restless and Wild                      Accept   
4   Let There Be Rock                      AC/DC    
5   Big Ones                               Aerosmith


#### Tracks to Albums, Genres, and MediaType

In [43]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
select *
from tracks
limit 10;

TrackId  Name                                     AlbumId  MediaTypeId  GenreId  Composer                                                                Milliseconds  Bytes     UnitPrice
-------  ---------------------------------------  -------  -----------  -------  ----------------------------------------------------------------------  ------------  --------  ---------
1        For Those About To Rock (We Salute You)  1        1            1        Angus Young, Malcolm Young, Brian Johnson                               343719        11170334  0.99     
2        Balls to the Wall                        2        2            1                                                                                342562        5510424   0.99     
3        Fast As a Shark                          3        2            1        F. Baltes, S. Kaufman, U. Dirkscneider & W. Hoffman                     230619        3990994   0.99     
4        Restless and Wild                        3        2     

In [54]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
select t.trackid, t.name, a.albumid, a.title
from tracks as t
inner join albums as a
on t.albumid = a.albumid
order by t.trackid
limit 10;

TrackId  Name                                     AlbumId  Title                                
-------  ---------------------------------------  -------  -------------------------------------
1        For Those About To Rock (We Salute You)  1        For Those About To Rock We Salute You
2        Balls to the Wall                        2        Balls to the Wall                    
3        Fast As a Shark                          3        Restless and Wild                    
4        Restless and Wild                        3        Restless and Wild                    
5        Princess of the Dawn                     3        Restless and Wild                    
6        Put The Finger On You                    1        For Those About To Rock We Salute You
7        Let's Get It Up                          1        For Those About To Rock We Salute You
8        Inject The Venom                         1        For Those About To Rock We Salute You
9        Snowballed           

In [52]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
select t.trackid, t.name, g.genreid, g.name
from tracks as t
inner join genres as g
on t.genreid = g.genreid
order by t.trackid
limit 10;

TrackId  Name                                     GenreId  Name
-------  ---------------------------------------  -------  ----
1        For Those About To Rock (We Salute You)  1        Rock
2        Balls to the Wall                        1        Rock
3        Fast As a Shark                          1        Rock
4        Restless and Wild                        1        Rock
5        Princess of the Dawn                     1        Rock
6        Put The Finger On You                    1        Rock
7        Let's Get It Up                          1        Rock
8        Inject The Venom                         1        Rock
9        Snowballed                               1        Rock
10       Evil Walks                               1        Rock


In [51]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
select t.trackid, t.name, m.mediatypeid, m.name
from tracks as t
inner join media_types as m
on t.mediatypeid = m.mediatypeid
order by t.trackid
limit 10;

TrackId  Name                                     MediaTypeId  Name                    
-------  ---------------------------------------  -----------  ------------------------
1        For Those About To Rock (We Salute You)  1            MPEG audio file         
2        Balls to the Wall                        2            Protected AAC audio file
3        Fast As a Shark                          2            Protected AAC audio file
4        Restless and Wild                        2            Protected AAC audio file
5        Princess of the Dawn                     2            Protected AAC audio file
6        Put The Finger On You                    1            MPEG audio file         
7        Let's Get It Up                          1            MPEG audio file         
8        Inject The Venom                         1            MPEG audio file         
9        Snowballed                               1            MPEG audio file         
10       Evil Walks             

#### Tracks Combined (CTEs)

In [60]:
# Select tracks and join with other tables
%%script sqlite3 --column --header chinook.db
with
a_a as (
select al.albumid, al.title, ar.artistid, ar.name
from albums as al
inner join artists as ar
on al.artistid = ar.artistid
),
t_a as (
select t.trackid as track, t.name as title, a.title as album, a.name as artist
from tracks as t
inner join a_a as a
on t.albumid = a.albumid
order by t.trackid
),
t_g as (
select t.trackid as track, t.name as title, g.name as genre
from tracks as t
inner join genres as g
on t.genreid = g.genreid
order by t.trackid
),
t_m as (
select t.trackid as track, t.name as title, m.name as media
from tracks as t
inner join media_types as m
on t.mediatypeid = m.mediatypeid
order by t.trackid
),
t_a_g as (
select t_a.track, t_a.title, t_a.album, t_a.artist, t_g.genre
from t_a
inner join t_g
on t_a.track = t_g.track
order by t_a.track
),
t_a_g_m as (
select t_a_g.track, t_a_g.title, t_a_g.album, t_a_g.artist, t_a_g.genre, t_m.media
from t_a_g
inner join t_m
on t_a_g.track = t_m.track
order by t_a_g.track
)
select *
from t_a_g_m
limit 10;

track  title                                    album                                  artist  genre  media                   
-----  ---------------------------------------  -------------------------------------  ------  -----  ------------------------
1      For Those About To Rock (We Salute You)  For Those About To Rock We Salute You  AC/DC   Rock   MPEG audio file         
2      Balls to the Wall                        Balls to the Wall                      Accept  Rock   Protected AAC audio file
3      Fast As a Shark                          Restless and Wild                      Accept  Rock   Protected AAC audio file
4      Restless and Wild                        Restless and Wild                      Accept  Rock   Protected AAC audio file
5      Princess of the Dawn                     Restless and Wild                      Accept  Rock   Protected AAC audio file
6      Put The Finger On You                    For Those About To Rock We Salute You  AC/DC   Rock   MPEG audi

#### Track, Invoice to InvoiceLine

In [21]:
# Relate tracks to invoice_items by trackId
%%script sqlite3 --column --header chinook.db
select il.invoicelineid, il.invoiceid, t.trackid, t.name, il.quantity, il.unitprice
from tracks as t
inner join invoice_items as il
on t.trackid = il.trackid
order by il.invoicelineid
limit 10;

InvoiceLineId  InvoiceId  TrackId  Name                   Quantity  UnitPrice
-------------  ---------  -------  ---------------------  --------  ---------
1              1          2        Balls to the Wall      1         0.99     
2              1          4        Restless and Wild      1         0.99     
3              2          6        Put The Finger On You  1         0.99     
4              2          8        Inject The Venom       1         0.99     
5              2          10       Evil Walks             1         0.99     
6              2          12       Breaking The Rules     1         0.99     
7              3          16       Dog Eat Dog            1         0.99     
8              3          20       Overdose               1         0.99     
9              3          24       Love In An Elevator    1         0.99     
10             3          28       Janie's Got A Gun      1         0.99     


In [23]:
# Relate invoices to invoice_items by invoiceId
%%script sqlite3 --column --header chinook.db
select il.invoicelineid, il.invoiceid, i.invoiceid, i.invoicedate, il.unitprice, il.quantity, i.total
from invoices as i
inner join invoice_items as il
on i.invoiceid = il.invoiceid
order by i.invoiceid
limit 10 ;

InvoiceLineId  InvoiceId  InvoiceId  InvoiceDate          UnitPrice  Quantity  Total
-------------  ---------  ---------  -------------------  ---------  --------  -----
1              1          1          2009-01-01 00:00:00  0.99       1         1.98 
2              1          1          2009-01-01 00:00:00  0.99       1         1.98 
3              2          2          2009-01-02 00:00:00  0.99       1         3.96 
4              2          2          2009-01-02 00:00:00  0.99       1         3.96 
5              2          2          2009-01-02 00:00:00  0.99       1         3.96 
6              2          2          2009-01-02 00:00:00  0.99       1         3.96 
7              3          3          2009-01-03 00:00:00  0.99       1         5.94 
8              3          3          2009-01-03 00:00:00  0.99       1         5.94 
9              3          3          2009-01-03 00:00:00  0.99       1         5.94 
10             3          3          2009-01-03 00:00:00  0.99   

In [26]:
# Relate invoices to invoice_items by invoiceId
%%script sqlite3 --column --header chinook.db
select il.invoiceid, SUM(il.unitprice * il.quantity) as calc_total, i.total
from invoices as i
inner join invoice_items as il
on i.invoiceid = il.invoiceid
group by il.invoiceid
order by i.invoiceid
limit 10 ;

InvoiceId  calc_total  Total
---------  ----------  -----
1          1.98        1.98 
2          3.96        3.96 
3          5.94        5.94 
4          8.91        8.91 
5          13.86       13.86
6          0.99        0.99 
7          1.98        1.98 
8          1.98        1.98 
9          3.96        3.96 
10         5.94        5.94 


In [36]:
# Relate tracks to invoice_items by trackId
%%script sqlite3 --column --header chinook.db
with
t_il as (
select t.trackid, t.name, il.invoicelineid, il.invoiceid
from tracks as t
inner join invoice_items as il
on t.trackid = il.trackid
order by t.trackid
),
i_il as (
select il.invoicelineid, il.invoiceid, i.invoiceid, i.customerid, i.invoicedate
from invoices as i
inner join invoice_items as il
on i.invoiceid = il.invoiceid
order by i.invoiceid
),
t_i_il as (
select i_il.invoicelineid, i_il.invoiceid, i_il.customerid, i_il.invoicedate, t_il.name as track
from i_il
inner join t_il
on i_il.invoicelineid = t_il.invoicelineid
order by i_il.invoicelineid
),
t_i_il_c as (
select t_i_il.invoicelineid, t_i_il.invoiceid, t_i_il.customerid, t_i_il.invoicedate, t_i_il.track, c.LastName, c.FirstName
from t_i_il
inner join customers as c
on t_i_il.customerid = c.customerid
order by t_i_il.invoicelineid
)
select *
from t_i_il_c
limit 10 ;

invoicelineid  invoiceid  customerid  invoicedate          track                  LastName  FirstName
-------------  ---------  ----------  -------------------  ---------------------  --------  ---------
1              1          2           2009-01-01 00:00:00  Balls to the Wall      Köhler    Leonie   
2              1          2           2009-01-01 00:00:00  Restless and Wild      Köhler    Leonie   
3              2          4           2009-01-02 00:00:00  Put The Finger On You  Hansen    Bjørn    
4              2          4           2009-01-02 00:00:00  Inject The Venom       Hansen    Bjørn    
5              2          4           2009-01-02 00:00:00  Evil Walks             Hansen    Bjørn    
6              2          4           2009-01-02 00:00:00  Breaking The Rules     Hansen    Bjørn    
7              3          8           2009-01-03 00:00:00  Dog Eat Dog            Peeters   Daan     
8              3          8           2009-01-03 00:00:00  Overdose               

#### Track to Playlist via PlaylistTrack

In [46]:
# Relate playlists to tracks
%%script sqlite3 --column --header chinook.db
with
t_pt as (
select t.trackid, t.name, t.composer, pt.playlistid
from tracks as t
inner join playlist_track as pt
on t.trackid = pt.trackid
order by t.trackid
),
t_pt_p as (
select t_pt.trackid as track, t_pt.name as title, t_pt.composer, t_pt.playlistid, p.name as playlist
from t_pt
inner join playlists as p
on t_pt.playlistid = p.playlistid
order by t_pt.trackid
)
select *
from t_pt_p
limit 10 ;

track  title                                    composer                                             playlistid  playlist           
-----  ---------------------------------------  ---------------------------------------------------  ----------  -------------------
1      For Those About To Rock (We Salute You)  Angus Young, Malcolm Young, Brian Johnson            1           Music              
1      For Those About To Rock (We Salute You)  Angus Young, Malcolm Young, Brian Johnson            8           Music              
1      For Those About To Rock (We Salute You)  Angus Young, Malcolm Young, Brian Johnson            17          Heavy Metal Classic
2      Balls to the Wall                                                                             1           Music              
2      Balls to the Wall                                                                             8           Music              
2      Balls to the Wall                                             

## Phase 3. Answer Guided Questions



Once you know the syntax and understand the data structure, you will put your skills to work solving specific business problems. Below are sets of explicit questions to answer using SQL. This phase mimics standard reporting: you are handed a specific prompt, and your job is to translate that prompt into an accurate, data-driven answer.

Here are some practice questions:
- https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/README.md
- [Using the R language]( https://rpubs.com/enext777/636199 )
- [Search Google]( https://www.google.com/search?q=chinook+database+questions )
- Ask AI to generate some questions for you.  Be sure to mention the Chinook data set.




## Phase 4. Ask and Answer Your Own Questions


This final phase is the most critical part of the project. The true hallmark of a great data scientist isn't just answering the questions you're handed; it's having the curiosity and intuition to formulate your own questions.

You are tasked with looking at the Chinook dataset, generating your own original hypotheses or business questions, and writing the SQL required to answer them.

Demonstrate that you know SQL fundamentals by posting a question for each SQL statement below and then using the SQL statement to answer you question.

You will use the material from this phase for the presentation.



## Why This Matters: The Path to Your Capstone


Learning to ask your own questions is the exact skill you will rely on for your final capstone project. In your capstone, you will choose a dataset from your own field of interest. No one will hand you a list of prompts or a guide telling you what to look for.

Use this project as your sandbox. Mastering the ability to interrogate a database now will give you the framework you need to discover meaningful, independent insights in your capstone later.



## Chinook data set

See the lecture on SQLite3 using the Chinook data set to set up the software, database, and tables, as well as for the links to ancillary information about the data set.


In [438]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


## Use SQL queries to pull specific information

Do NOT pull all the data and then filter using DataFrame methods.  The initial purpose is to get familiar with SQL.




### Basic Queries


#### SELECT (with * and with column trackids)


#### WHERE


#### AND


#### OR


#### LIKE (with % or _ wildcard)


#### BETWEEN


#### LIMIT



### Sorting and Grouping


#### ORDER BY


#### DISTINCT


#### GROUP BY



### Aggregates


#### MAX


#### MIN


#### SUM


#### AVG


#### COUNT



## Make some plots

Make some cool plots to go with your data. Write SQL queries to get ONLY the information you need for each plot. (Don't pull ALL the data and then just plot a few columns.)



## EXTRA CREDIT:
* Use a CTE
* Use a query that joins two or more tables.
* Make a model to see if you can predict something ( e.g. GNB classifier, linear regression )
* Come up with something else cool to do with your data
